# Assignment 3 · Risk and prediction with time series
16 points (16%); release 10 November 2026; due 24 November 2026, 23:59 KST.

**목적 / Goal.** 모니터링 담당자에게 다음 관측값의 예상과 움직임의 불확실성을 구분해 보고합니다. GARCH는 실제 시장 수익률의 퍼짐, VAR는 두 모의 관측 신호의 예측 정보를 다룹니다. 칼만 필터(Q5)는 잡음 섞인 모의 센서로 숨은 수준을 따라갑니다. 서로 다른 자료이며 하나의 결합 모형으로 연결하지 않습니다.

Submit exactly TWO files: (1) a PDF containing Q1–Q5 answers and the Analysis Receipt block as selectable text, not a screenshot; (2) your actual IPYNB containing Code 1–6 and saved outputs. Run all, save, then use Colab File > Download > Download .ipynb. The choices and numbers must agree. Typical written length is 1–3 pages excluding the receipt, not a strict limit. Typed or handwritten answers are accepted; the receipt must be selectable text. Figures are optional.
All six automated checks passing earns 1 coding-completion point; otherwise 0. A code line that does not pass never blocks a later step, and supported Q1–Q5 answers earn partial credit out of 15. Submit both files even with NOT READY. No separate code-correction period or post-deadline revision of an already-submitted assignment. Late FIRST submissions lose 10% of earned marks per day, up to five days. Post notebook questions to the LMS Q&A board at least 48 hours before the deadline. Disclose AI tools, purpose and scope, or write “none”.

**Scope:** Weeks 9–11. GARCH (Q1), VAR (Q2–Q4) and the Kalman filter (Q5). Q5 uses the 10 November class, so it comes last.

**처음 시작 / Start here:** 회색으로 제공된 코드는 읽거나 고치지 않아도 됩니다. 처음에는 모두 실행해 화면 위치를 확인하세요. None 또는 np.nan가 있는 여섯 줄만 수정합니다. 변수명, 함수, 반복문, 표 만드는 코드는 제공됩니다. `**2`는 제곱, `@`는 행렬곱, `np.mean(...)`은 평균, `np.sqrt(...)`는 제곱근입니다. 오류가 나면 우선 해당 줄만 원래 빈칸으로 되돌리세요. 질문마다 요청한 Q*_READ/Q4_SELECT/Q4_FINAL 출력만 읽으면 됩니다. 고급 출력은 채점하지 않습니다. / Edit six marked lines only. Helpers and loops are supplied.

**Workflow:** Run all → Q1–Q3 with Code 1–5 → inspect validation → choose VAR1 or VAR2 and enter reason → reveal → Q4(b) with Code 6 → Q5 after the Week 11 class → AI disclosure/receipt → save outputs. A code line that does not pass never blocks the next step. CPU only; the first cell installs the course package versions, which needs internet; the data are inside the notebook. Nine short written prompts and six one-line blanks; Q5 needs no code.

**자료 규칙 / Data rule:** final observations are embedded and technically inspectable. The gate and local commitment file prevent ordinary accidental changes only. Treat them as unused for YOUR decisions; never claim a sealed blind test. Restarting a Colab runtime may erase the local lock. Preserve the commitment shown in your saved outputs.

In [ ]:
#@title 준비 / Setup
import sys, subprocess, importlib.metadata as md
for package, version in [('arch','8.0.0'),('statsmodels','0.15.0')]:
    try: current = md.version(package)
    except md.PackageNotFoundError: current = None
    if current != version:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package+'=='+version])
import json, hashlib, platform, datetime, warnings, inspect
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arch, statsmodels
from arch import arch_model
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ar_model import AutoReg
plt.rcParams.update({'figure.figsize':(10,3),'axes.grid':True,'grid.alpha':.25})
SHOW_DETAILS=False # Optional reference tables; no extra credit.
CHECKS={}; CHECK_SIGNATURES={}; RESULTS={}; AUDIT=[]; HOLD=None; OPENED=False
RUN_ID=datetime.datetime.now(datetime.timezone.utc).isoformat()
def digest(v): return hashlib.sha256(json.dumps(v,sort_keys=True,default=str).encode()).hexdigest()
# p값은 0.00000 대신 < 0.001 로 / p-values below 0.001 print as < 0.001, not 0.00000
P_COLUMNS=('residual_p','x_to_y_Granger_p','white_p','p_value','ADF_p','KPSS_p')
def _fmt_p(v):
    try: v=float(v)
    except (TypeError,ValueError): return str(v)
    return '< 0.001' if v<0.001 else f'{v:.3f}'
def table(name,df):
    if SHOW_DETAILS or name.startswith(('Q1_READ','Q2_READ','Q3_READ','Q4_SELECT','Q4_FINAL','Q5_READ')):
        pcols={c:_fmt_p for c in df.columns if c in P_COLUMNS or (c=='p' and 'series' in df.columns)}
        print(name); print(df.to_string(float_format=lambda v:f'{v:.5f}',formatters=pcols))
    RESULTS[name]=df.to_json(orient='split',double_precision=8)
def check(number,fn,cases):
    ok=True
    try:
        for args,want in cases:
            got=np.asarray(fn(*args),dtype=float)
            target=np.asarray(want,dtype=float)
            if got.shape!=target.shape or not np.all(np.isfinite(got)) or not np.allclose(got,target,rtol=1e-6,atol=1e-8):
                raise ValueError('값·모양·단위를 확인하세요 / check value, shape and units')
    except Exception as exc:
        ok=False; print(f'Code {number}: 미완 / NOT PASSED ({type(exc).__name__}: {exc})')
    if ok: print(f'Code {number}: PASS')
    CHECKS[number]=ok
    CHECK_SIGNATURES[number]=hashlib.sha256(fn.__code__.co_code+repr(fn.__code__.co_consts).encode()).hexdigest()[:16]
    return ok
def acf_panel(values,titles):
    fig,axes=plt.subplots(1,len(values),figsize=(4*len(values),2.8),squeeze=False)
    for ax,v,title in zip(axes[0],values,titles):
        a=acf(np.asarray(v),nlags=20,fft=False)[1:]
        ax.bar(range(1,21),a,color='#285943'); ax.axhline(0,color='black',lw=.5)
        ax.axhline(2/np.sqrt(len(v)),ls='--',color='#b38b2d');ax.axhline(-2/np.sqrt(len(v)),ls='--',color='#b38b2d')
        ax.set(title=title,xlabel='Lag')
    plt.tight_layout();plt.show()
print('CPU | arch',arch.__version__,'| statsmodels',statsmodels.__version__)


In [ ]:
#@title 자료 / Embedded data. Do not inspect final observations before commitment.
DATA_TEXT = '{"garch":{"date":["1999-01-05","1999-01-06","1999-01-07","1999-01-08","1999-01-11","1999-01-12","1999-01-13","1999-01-14","1999-01-15","1999-01-19","1999-01-20","1999-01-21","1999-01-22","1999-01-25","1999-01-26","1999-01-27","1999-01-28","1999-01-29","1999-02-01","1999-02-02","1999-02-03","1999-02-04","1999-02-05","1999-02-08","1999-02-09","1999-02-10","1999-02-11","1999-02-12","1999-02-16","1999-02-17","1999-02-18","1999-02-19","1999-02-22","1999-02-23","1999-02-24","1999-02-25","1999-02-26","1999-03-01","1999-03-02","1999-03-03","1999-03-04","1999-03-05","1999-03-08","1999-03-09","1999-03-10","1999-03-11","1999-03-12","1999-03-15","1999-03-16","1999-03-17","1999-03-18","1999-03-19","1999-03-22","1999-03-23","1999-03-24","1999-03-25","1999-03-26","1999-03-29","1999-03-30","1999-03-31","1999-04-01","1999-04-05","1999-04-06","1999-04-07","1999-04-08","1999-04-09","1999-04-12","1999-04-13","1999-04-14","1999-04-15","1999-04-16","1999-04-19","1999-04-20","1999-04-21","1999-04-22","1999-04-23","1999-04-26","1999-04-27","1999-04-28","1999-04-29","1999-04-30","1999-05-03","1999-05-04","1999-05-05","1999-05-06","1999-05-07","1999-05-10","1999-05-11","1999-05-12","1999-05-13","1999-05-14","1999-05-17","1999-05-18","1999-05-19","1999-05-20","1999-05-21","1999-05-24","1999-05-25","1999-05-26","1999-05-27","1999-05-28","1999-06-01","1999-06-02","1999-06-03","1999-06-04","1999-06-07","1999-06-08","1999-06-09","1999-06-10","1999-06-11","1999-06-14","1999-06-15","1999-06-16","1999-06-17","1999-06-18","1999-06-21","1999-06-22","1999-06-23","1999-06-24","1999-06-25","1999-06-28","1999-06-29","1999-06-30","1999-07-01","1999-07-02","1999-07-06","1999-07-07","1999-07-08","1999-07-09","1999-07-12","1999-07-13","1999-07-14","1999-07-15","1999-07-16","1999-07-19","1999-07-20","1999-07-21","1999-07-22","1999-07-23","1999-07-26","1999-07-27","1999-07-28","1999-07-29","1999-07-30","1999-08-02","1999-08-03","1999-08-04","1999-08-05","1999-08-06","1999-08-09","1999-08-10","1999-08-11","1999-08-12","1999-08-13","1999-08-16","1999-08-17","1999-08-18","1999-08-19","1999-08-20","1999-08-23","1999-08-24","1999-08-25","1999-08-26","1999-08-27","1999-08-30","1999-08-31","1999-09-01","1999-09-02","1999-09-03","1999-09-07","1999-09-08","1999-09-09","1999-09-10","1999-09-13","1999-09-14","1999-09-15","1999-09-16","1999-09-17","1999-09-20","1999-09-21","1999-09-22","1999-09-23","1999-09-24","1999-09-27","1999-09-28","1999-09-29","1999-09-30","1999-10-01","1999-10-04","1999-10-05","1999-10-06","1999-10-07","1999-10-08","1999-10-11","1999-10-12","1999-10-13","1999-10-14","1999-10-15","1999-10-18","1999-10-19","1999-10-20","1999-10-21","1999-10-22","1999-10-25","1999-10-26","1999-10-27","1999-10-28","1999-10-29","1999-11-01","1999-11-02","1999-11-03","1999-11-04","1999-11-05","1999-11-08","1999-11-09","1999-11-10","1999-11-11","1999-11-12","1999-11-15","1999-11-16","1999-11-17","1999-11-18","1999-11-19","1999-11-22","1999-11-23","1999-11-24","1999-11-26","1999-11-29","1999-11-30","1999-12-01","1999-12-02","1999-12-03","1999-12-06","1999-12-07","1999-12-08","1999-12-09","1999-12-10","1999-12-13","1999-12-14","1999-12-15","1999-12-16","1999-12-17","1999-12-20","1999-12-21","1999-12-22","1999-12-23","1999-12-27","1999-12-28","1999-12-29","1999-12-30","1999-12-31","2000-01-03","2000-01-04","2000-01-05","2000-01-06","2000-01-07","2000-01-10","2000-01-11","2000-01-12","2000-01-13","2000-01-14","2000-01-18","2000-01-19","2000-01-20","2000-01-21","2000-01-24","2000-01-25","2000-01-26","2000-01-27","2000-01-28","2000-01-31","2000-02-01","2000-02-02","2000-02-03","2000-02-04","2000-02-07","2000-02-08","2000-02-09","2000-02-10","2000-02-11","2000-02-14","2000-02-15","2000-02-16","2000-02-17","2000-02-18","2000-02-22","2000-02-23","2000-02-24","2000-02-25","2000-02-28","2000-02-29","2000-03-01","2000-03-02","2000-03-03","2000-03-06","2000-03-07","2000-03-08","2000-03-09","2000-03-10","2000-03-13","2000-03-14","2000-03-15","2000-03-16","2000-03-17","2000-03-20","2000-03-21","2000-03-22","2000-03-23","2000-03-24","2000-03-27","2000-03-28","2000-03-29","2000-03-30","2000-03-31","2000-04-03","2000-04-04","2000-04-05","2000-04-06","2000-04-07","2000-04-10","2000-04-11","2000-04-12","2000-04-13","2000-04-14","2000-04-17","2000-04-18","2000-04-19","2000-04-20","2000-04-24","2000-04-25","2000-04-26","2000-04-27","2000-04-28","2000-05-01","2000-05-02","2000-05-03","2000-05-04","2000-05-05","2000-05-08","2000-05-09","2000-05-10","2000-05-11","2000-05-12","2000-05-15","2000-05-16","2000-05-17","2000-05-18","2000-05-19","2000-05-22","2000-05-23","2000-05-24","2000-05-25","2000-05-26","2000-05-30","2000-05-31","2000-06-01","2000-06-02","2000-06-05","2000-06-06","2000-06-07","2000-06-08","2000-06-09","2000-06-12","2000-06-13","2000-06-14","2000-06-15","2000-06-16","2000-06-19","2000-06-20","2000-06-21","2000-06-22","2000-06-23","2000-06-26","2000-06-27","2000-06-28","2000-06-29","2000-06-30","2000-07-03","2000-07-05","2000-07-06","2000-07-07","2000-07-10","2000-07-11","2000-07-12","2000-07-13","2000-07-14","2000-07-17","2000-07-18","2000-07-19","2000-07-20","2000-07-21","2000-07-24","2000-07-25","2000-07-26","2000-07-27","2000-07-28","2000-07-31","2000-08-01","2000-08-02","2000-08-03","2000-08-04","2000-08-07","2000-08-08","2000-08-09","2000-08-10","2000-08-11","2000-08-14","2000-08-15","2000-08-16","2000-08-17","2000-08-18","2000-08-21","2000-08-22","2000-08-23","2000-08-24","2000-08-25","2000-08-28","2000-08-29","2000-08-30","2000-08-31","2000-09-01","2000-09-05","2000-09-06","2000-09-07","2000-09-08","2000-09-11","2000-09-12","2000-09-13","2000-09-14","2000-09-15","2000-09-18","2000-09-19","2000-09-20","2000-09-21","2000-09-22","2000-09-25","2000-09-26","2000-09-27","2000-09-28","2000-09-29","2000-10-02","2000-10-03","2000-10-04","2000-10-05","2000-10-06","2000-10-09","2000-10-10","2000-10-11","2000-10-12","2000-10-13","2000-10-16","2000-10-17","2000-10-18","2000-10-19","2000-10-20","2000-10-23","2000-10-24","2000-10-25","2000-10-26","2000-10-27","2000-10-30","2000-10-31","2000-11-01","2000-11-02","2000-11-03","2000-11-06","2000-11-07","2000-11-08","2000-11-09","2000-11-10","2000-11-13","2000-11-14","2000-11-15","2000-11-16","2000-11-17","2000-11-20","2000-11-21","2000-11-22","2000-11-24","2000-11-27","2000-11-28","2000-11-29","2000-11-30","2000-12-01","2000-12-04","2000-12-05","2000-12-06","2000-12-07","2000-12-08","2000-12-11","2000-12-12","2000-12-13","2000-12-14","2000-12-15","2000-12-18","2000-12-19","2000-12-20","2000-12-21","2000-12-22","2000-12-26","2000-12-27","2000-12-28","2000-12-29","2001-01-02","2001-01-03","2001-01-04","2001-01-05","2001-01-08","2001-01-09","2001-01-10","2001-01-11","2001-01-12","2001-01-16","2001-01-17","2001-01-18","2001-01-19","2001-01-22","2001-01-23","2001-01-24","2001-01-25","2001-01-26","2001-01-29","2001-01-30","2001-01-31","2001-02-01","2001-02-02","2001-02-05","2001-02-06","2001-02-07","2001-02-08","2001-02-09","2001-02-12","2001-02-13","2001-02-14","2001-02-15","2001-02-16","2001-02-20","2001-02-21","2001-02-22","2001-02-23","2001-02-26","2001-02-27","2001-02-28","2001-03-01","2001-03-02","2001-03-05","2001-03-06","2001-03-07","2001-03-08","2001-03-09","2001-03-12","2001-03-13","2001-03-14","2001-03-15","2001-03-16","2001-03-19","2001-03-20","2001-03-21","2001-03-22","2001-03-23","2001-03-26","2001-03-27","2001-03-28","2001-03-29","2001-03-30","2001-04-02","2001-04-03","2001-04-04","2001-04-05","2001-04-06","2001-04-09","2001-04-10","2001-04-11","2001-04-12","2001-04-16","2001-04-17","2001-04-18","2001-04-19","2001-04-20","2001-04-23","2001-04-24","2001-04-25","2001-04-26","2001-04-27","2001-04-30","2001-05-01","2001-05-02","2001-05-03","2001-05-04","2001-05-07","2001-05-08","2001-05-09","2001-05-10","2001-05-11","2001-05-14","2001-05-15","2001-05-16","2001-05-17","2001-05-18","2001-05-21","2001-05-22","2001-05-23","2001-05-24","2001-05-25","2001-05-29","2001-05-30","2001-05-31","2001-06-01","2001-06-04","2001-06-05","2001-06-06","2001-06-07","2001-06-08","2001-06-11","2001-06-12","2001-06-13","2001-06-14","2001-06-15","2001-06-18","2001-06-19","2001-06-20","2001-06-21","2001-06-22","2001-06-25","2001-06-26","2001-06-27","2001-06-28","2001-06-29","2001-07-02","2001-07-03","2001-07-05","2001-07-06","2001-07-09","2001-07-10","2001-07-11","2001-07-12","2001-07-13","2001-07-16","2001-07-17","2001-07-18","2001-07-19","2001-07-20","2001-07-23","2001-07-24","2001-07-25","2001-07-26","2001-07-27","2001-07-30","2001-07-31","2001-08-01","2001-08-02","2001-08-03","2001-08-06","2001-08-07","2001-08-08","2001-08-09","2001-08-10","2001-08-13","2001-08-14","2001-08-15","2001-08-16","2001-08-17","2001-08-20","2001-08-21","2001-08-22","2001-08-23","2001-08-24","2001-08-27","2001-08-28","2001-08-29","2001-08-30","2001-08-31","2001-09-04","2001-09-05","2001-09-06","2001-09-07","2001-09-10","2001-09-17","2001-09-18","2001-09-19","2001-09-20","2001-09-21","2001-09-24","2001-09-25","2001-09-26","2001-09-27","2001-09-28","2001-10-01","2001-10-02","2001-10-03","2001-10-04","2001-10-05","2001-10-08","2001-10-09","2001-10-10","2001-10-11","2001-10-12","2001-10-15","2001-10-16","2001-10-17","2001-10-18","2001-10-19","2001-10-22","2001-10-23","2001-10-24","2001-10-25","2001-10-26","2001-10-29","2001-10-30","2001-10-31","2001-11-01","2001-11-02","2001-11-05","2001-11-06","2001-11-07","2001-11-08","2001-11-09","2001-11-12","2001-11-13","2001-11-14","2001-11-15","2001-11-16","2001-11-19","2001-11-20","2001-11-21","2001-11-23","2001-11-26","2001-11-27","2001-11-28","2001-11-29","2001-11-30","2001-12-03","2001-12-04","2001-12-05","2001-12-06","2001-12-07","2001-12-10","2001-12-11","2001-12-12","2001-12-13","2001-12-14","2001-12-17","2001-12-18","2001-12-19","2001-12-20","2001-12-21","2001-12-24","2001-12-26","2001-12-27","2001-12-28","2001-12-31","2002-01-02","2002-01-03","2002-01-04","2002-01-07","2002-01-08","2002-01-09","2002-01-10","2002-01-11","2002-01-14","2002-01-15","2002-01-16","2002-01-17","2002-01-18","2002-01-22","2002-01-23","2002-01-24","2002-01-25","2002-01-28","2002-01-29","2002-01-30","2002-01-31","2002-02-01","2002-02-04","2002-02-05","2002-02-06","2002-02-07","2002-02-08","2002-02-11","2002-02-12","2002-02-13","2002-02-14","2002-02-15","2002-02-19","2002-02-20","2002-02-21","2002-02-22","2002-02-25","2002-02-26","2002-02-27","2002-02-28","2002-03-01","2002-03-04","2002-03-05","2002-03-06","2002-03-07","2002-03-08","2002-03-11","2002-03-12","2002-03-13","2002-03-14","2002-03-15","2002-03-18","2002-03-19","2002-03-20","2002-03-21","2002-03-22","2002-03-25","2002-03-26","2002-03-27","2002-03-28","2002-04-01","2002-04-02","2002-04-03","2002-04-04","2002-04-05","2002-04-08","2002-04-09","2002-04-10","2002-04-11","2002-04-12","2002-04-15","2002-04-16","2002-04-17","2002-04-18","2002-04-19","2002-04-22","2002-04-23","2002-04-24","2002-04-25","2002-04-26","2002-04-29","2002-04-30","2002-05-01","2002-05-02","2002-05-03","2002-05-06","2002-05-07","2002-05-08","2002-05-09","2002-05-10","2002-05-13","2002-05-14","2002-05-15","2002-05-16","2002-05-17","2002-05-20","2002-05-21","2002-05-22","2002-05-23","2002-05-24","2002-05-28","2002-05-29","2002-05-30","2002-05-31","2002-06-03","2002-06-04","2002-06-05","2002-06-06","2002-06-07","2002-06-10","2002-06-11","2002-06-12","2002-06-13","2002-06-14","2002-06-17","2002-06-18","2002-06-19","2002-06-20","2002-06-21","2002-06-24","2002-06-25","2002-06-26","2002-06-27","2002-06-28","2002-07-01","2002-07-02","2002-07-03","2002-07-05","2002-07-08","2002-07-09","2002-07-10","2002-07-11","2002-07-12","2002-07-15","2002-07-16","2002-07-17","2002-07-18","2002-07-19","2002-07-22","2002-07-23","2002-07-24","2002-07-25","2002-07-26","2002-07-29","2002-07-30","2002-07-31","2002-08-01","2002-08-02","2002-08-05","2002-08-06","2002-08-07","2002-08-08","2002-08-09","2002-08-12","2002-08-13","2002-08-14","2002-08-15","2002-08-16","2002-08-19","2002-08-20","2002-08-21","2002-08-22","2002-08-23","2002-08-26","2002-08-27","2002-08-28","2002-08-29","2002-08-30","2002-09-03","2002-09-04","2002-09-05","2002-09-06","2002-09-09","2002-09-10","2002-09-11","2002-09-12","2002-09-13","2002-09-16","2002-09-17","2002-09-18","2002-09-19","2002-09-20","2002-09-23","2002-09-24","2002-09-25","2002-09-26","2002-09-27","2002-09-30","2002-10-01","2002-10-02","2002-10-03","2002-10-04","2002-10-07","2002-10-08","2002-10-09","2002-10-10","2002-10-11","2002-10-14","2002-10-15","2002-10-16","2002-10-17","2002-10-18","2002-10-21","2002-10-22","2002-10-23","2002-10-24","2002-10-25","2002-10-28","2002-10-29","2002-10-30","2002-10-31","2002-11-01","2002-11-04","2002-11-05","2002-11-06","2002-11-07","2002-11-08","2002-11-11","2002-11-12","2002-11-13","2002-11-14","2002-11-15","2002-11-18","2002-11-19","2002-11-20","2002-11-21","2002-11-22","2002-11-25","2002-11-26","2002-11-27","2002-11-29","2002-12-02","2002-12-03","2002-12-04","2002-12-05","2002-12-06","2002-12-09","2002-12-10","2002-12-11","2002-12-12","2002-12-13","2002-12-16","2002-12-17","2002-12-18","2002-12-19","2002-12-20","2002-12-23","2002-12-24","2002-12-26","2002-12-27","2002-12-30","2002-12-31","2003-01-02","2003-01-03","2003-01-06","2003-01-07","2003-01-08","2003-01-09","2003-01-10","2003-01-13","2003-01-14","2003-01-15","2003-01-16","2003-01-17","2003-01-21","2003-01-22","2003-01-23","2003-01-24","2003-01-27","2003-01-28","2003-01-29","2003-01-30","2003-01-31","2003-02-03","2003-02-04","2003-02-05","2003-02-06","2003-02-07","2003-02-10","2003-02-11","2003-02-12","2003-02-13","2003-02-14","2003-02-18","2003-02-19","2003-02-20","2003-02-21","2003-02-24","2003-02-25","2003-02-26","2003-02-27","2003-02-28","2003-03-03","2003-03-04","2003-03-05","2003-03-06","2003-03-07","2003-03-10","2003-03-11","2003-03-12","2003-03-13","2003-03-14","2003-03-17","2003-03-18","2003-03-19","2003-03-20","2003-03-21","2003-03-24","2003-03-25","2003-03-26","2003-03-27","2003-03-28","2003-03-31","2003-04-01","2003-04-02","2003-04-03","2003-04-04","2003-04-07","2003-04-08","2003-04-09","2003-04-10","2003-04-11","2003-04-14","2003-04-15","2003-04-16","2003-04-17","2003-04-21","2003-04-22","2003-04-23","2003-04-24","2003-04-25","2003-04-28","2003-04-29","2003-04-30","2003-05-01","2003-05-02","2003-05-05","2003-05-06","2003-05-07","2003-05-08","2003-05-09","2003-05-12","2003-05-13","2003-05-14","2003-05-15","2003-05-16","2003-05-19","2003-05-20","2003-05-21","2003-05-22","2003-05-23","2003-05-27","2003-05-28","2003-05-29","2003-05-30","2003-06-02","2003-06-03","2003-06-04","2003-06-05","2003-06-06","2003-06-09","2003-06-10","2003-06-11","2003-06-12","2003-06-13","2003-06-16","2003-06-17","2003-06-18","2003-06-19","2003-06-20","2003-06-23","2003-06-24","2003-06-25","2003-06-26","2003-06-27","2003-06-30","2003-07-01","2003-07-02","2003-07-03","2003-07-07","2003-07-08","2003-07-09","2003-07-10","2003-07-11","2003-07-14","2003-07-15","2003-07-16","2003-07-17","2003-07-18","2003-07-21","2003-07-22","2003-07-23","2003-07-24","2003-07-25","2003-07-28","2003-07-29","2003-07-30","2003-07-31","2003-08-01","2003-08-04","2003-08-05","2003-08-06","2003-08-07","2003-08-08","2003-08-11","2003-08-12","2003-08-13","2003-08-14","2003-08-15","2003-08-18","2003-08-19","2003-08-20","2003-08-21","2003-08-22","2003-08-25","2003-08-26","2003-08-27","2003-08-28","2003-08-29","2003-09-02","2003-09-03","2003-09-04","2003-09-05","2003-09-08","2003-09-09","2003-09-10","2003-09-11","2003-09-12","2003-09-15","2003-09-16","2003-09-17","2003-09-18","2003-09-19","2003-09-22","2003-09-23","2003-09-24","2003-09-25","2003-09-26","2003-09-29","2003-09-30","2003-10-01","2003-10-02","2003-10-03","2003-10-06","2003-10-07","2003-10-08","2003-10-09","2003-10-10","2003-10-13","2003-10-14","2003-10-15","2003-10-16","2003-10-17","2003-10-20","2003-10-21","2003-10-22","2003-10-23","2003-10-24","2003-10-27"],"return":[1.3490590680341086,2.1898867303733915,-0.20534343842202318,0.42124739979927384,-0.8830379224564311,-1.9470210167572333,-0.41311067880718255,-1.8156449144461462,2.5308377693282758,0.7005302619627507,0.3683300166929193,-1.7225028973706813,-0.8104658016497623,0.7148815834074895,1.4745191287230064,-0.7325288516597084,1.7699961120412055,1.1214234515622756,-0.5202480520104302,-0.8686487341905469,0.795562016643725,-1.8710639315397515,-0.7307402174325794,0.35196949334519445,-2.246518501363326,0.607458865006727,2.4613861248157676,-1.9250511807180182,0.9498445198605765,-1.446958683362265,1.07667280311734,0.15666807161700547,2.621840751242299,-0.07548881143879527,-1.4077781229665298,-0.6716254092968832,-0.5387947932444348,-0.17538341759157916,-0.8660904014278437,0.1793536306594845,1.5309483894939468,2.286277121068192,0.5675888225765036,-0.225556007145844,0.5454530787416978,0.8388522032475088,-0.23840799297829562,0.9739334909973252,-0.06733942151200267,-0.6574064964541826,1.432882376961242,-1.3196724501193025,-0.1756368326537583,-2.7252918240320056,0.509731580530115,1.6728437973714705,-0.5589231618997381,2.1111706334773217,-0.7215912181051998,-1.1116726800075583,0.5697472552711247,2.095809096140311,-0.24478737106878867,0.6805885281907287,1.2797476683147124,0.32462587491668415,0.759523786014249,-0.6505632543999162,-1.5965933155647427,-0.4216795188828115,-0.29146081423379044,-2.2634852913876458,1.28602037013259,2.2670663175322403,1.6846748627248331,-0.14508172216256554,0.23483206522039168,0.2027303004283887,-0.8762978153313838,-0.5999170122959363,-0.5713139815859414,1.4462203335738444,-1.6846785885467952,1.1428483676794166,-1.1390910058367787,0.9674867317178482,-0.3500506992510033,1.1358020404446556,0.6170032955531646,0.26066137583065085,-2.2001662850108517,0.12624267074716755,-0.4616905135565119,0.8149311632326572,-0.40252781675613747,-0.639907247116156,-1.7930364576524305,-1.7174926399465384,1.5727421560776556,-1.8058062789746643,1.581756156332137,-0.5839511274373876,0.04249008281860256,0.3646373898789079,2.147539461442971,0.5085911302904478,-1.2964764174232535,0.09939863672761007,-1.2069809412255772,-0.7071142546680953,0.027823422132566833,0.5518004341774407,2.223099054026001,0.7107811263466601,0.21917466380303452,0.45768283633789153,-0.9773322761684966,-0.21131587684530828,-1.304742797948233,-0.03572435677074992,1.2121022850215546,1.4984606863690964,1.5608805888736477,0.5992021497053024,0.7402157939806919,-0.22307291108099037,0.55603919551519,-0.10321122703533803,0.633378445747379,-0.2983218616037675,-0.396748920208001,0.3302603776211832,0.8155890796965437,0.6477206864245133,-0.7875703499488296,-2.194184301605784,0.1589080971081458,-1.3371242791764715,-0.29655382878486236,-0.6788158698952884,1.1126770245985185,0.18767108177941694,-1.8009446540180996,-0.92219460560381,-0.050431313900389796,-0.4429807504153338,-1.2826081677140166,0.6399317202638422,-1.0267819359511599,-0.1916785544786137,-1.2693876978741692,1.5871136394703278,-0.2899916937034952,2.248519539475602,0.2324638270787993,1.0011570823403204,-0.8457329453554152,-0.6964262279001154,0.9788830317880581,1.7509883536930282,0.240849325224346,1.332486502979524,-1.4418231146938076,-1.0139253183232277,-1.814972575671181,-0.27302580660322207,0.804076931687181,-0.9025834098171259,2.8495963169918426,-0.5015384612749862,-0.4675973084904861,0.2607919514185042,0.29637114733302994,-0.5586524525230452,-0.5849820721024912,-1.3804494555127889,0.0386890717321009,1.2766339289092876,0.008235646272414243,-2.11501716229332,0.22383151544538293,-2.3236016361718903,-0.23849293900841673,0.46472873166321094,-0.08654092675186575,-1.084470674001814,1.124238932866195,0.007803330262134978,1.6843431736697667,-0.24942932480307434,1.831215004434572,-0.5872049748169239,1.3852791000558362,-0.06065063888867073,-1.6743464499624494,-2.1158408688775054,-0.16582563525338045,-2.845899509338956,0.537268007702707,0.571663974289649,2.204155330949842,-0.4523893011937119,1.3956299755996326,-0.6180484910827566,-0.9101044222607158,1.1479077601905807,3.465855236401705,1.5148024001134353,-0.6485040552976606,-0.4722685034447949,0.5320724206476335,0.5674173243162883,0.555458878658488,0.4935894781571193,-0.8554932099182899,0.5973518007190926,0.5807807641294183,1.0513142925368868,-0.11969712090209583,1.8249073734490473,-0.6613032039604505,1.003657005883074,-0.20653347847021664,-0.0745748465410756,-1.1537530040984123,0.8817332494635011,-0.03246363496440807,-0.622426846342794,-1.3530189220071875,0.6323025544420879,0.8066332239756058,1.7070863013551651,-0.6980346397988946,-0.9998255927918365,-0.37610743810354563,0.3008533604743846,0.6321847452688267,-0.1285240934175036,-0.855098148892175,0.7214596888993619,0.38487768883008755,0.15846133543151453,-0.2071109397366655,1.0759339494139653,0.18817879683483696,1.5346779987413406,-0.08506366788116893,0.03842909913478465,0.39710348301813525,0.0689914078218834,0.32586840442760945,-0.9594994496268328,-3.9099175505866057,0.19203376717689125,0.09552218683284153,2.6729950058290264,1.1127824951257637,-1.3148576790573685,-0.43960199610104667,1.209624536481435,1.0614763015335704,-0.68555164525268,0.05221563476185409,-0.7120612485422839,-0.29165681778993147,-2.802258419637127,0.604648360431792,-0.4221619428148138,-0.3946203861979747,-2.7840813152317345,2.490485070605164,1.0571740659929496,-0.011356372325366237,1.1185347989420613,-0.04211333263324235,-0.009127609660541935,1.2198494043072827,-2.1035117221395616,0.36202427699052464,-2.1192299035074136,0.2030886645752794,0.8674947967946878,-1.0309372899858893,0.04250582668916536,-3.0847103117030095,0.4506673388070759,0.6281138767331385,-0.5349729804105863,-1.494008986267925,1.0957079137145165,1.3535073009169452,0.930211439390316,0.18617285991711086,1.9642853666568705,-1.2776702246402039,-2.5965296993352815,0.8140128567621474,2.527957023935379,-0.47340548570584673,-0.8241305110622932,-1.7843727395367992,2.398262033021492,4.654577881006983,0.4105461313924863,-0.5367830535901952,2.524451292159924,0.452162892218233,1.764249808132412,0.007200775322591113,-0.23596199245181637,-1.0641384486793193,0.052385579316549524,-1.3749849894415078,0.7138762612269645,0.49192258953869583,-0.749161445876112,-0.49361188462473393,0.9348563711456315,0.994809544823827,-0.7872111552905459,-0.25756624571871,-2.2522936257881376,-1.8338180696232165,-6.00450973852551,3.254811022398929,2.8260296846716493,-0.9856907930093861,0.49406412674652955,-0.3267740626521132,3.2734293511893497,-1.119653912905072,0.26863822317269026,-0.8562610514615265,1.0833162429883636,-1.506953157614177,-2.1801500828093623,-0.3915527985883571,1.6227264385197948,-0.5922700633593792,-0.8482923862157321,-2.0815054171820613,1.774410660944703,0.9297324724377276,2.185713714653481,0.9375105595749389,-1.2519718102749877,-0.7341489650408484,-2.1279500999946244,-0.44378366387167745,-1.9362084026941062,1.8169184477265787,-1.2609113111524906,-0.2536655983266911,3.173300178028171,-0.1301401880797748,1.9663269862960853,1.9446456774673493,-0.654016909081534,-0.669299336063478,0.9231268084488597,-0.660748520027532,-0.3234471855290799,-0.7544052165836312,1.6080211940976419,0.07483711173730256,0.5553890962352703,-0.9697050551745079,1.4601399090037326,-0.6786129228779636,0.21522634073622982,-1.8388167627191088,-0.7395563157870377,0.9548627201079185,-0.3276148252550293,0.2939315874758819,-0.8580674119660792,0.8429461846206188,1.0218520949274001,-1.598929645488667,0.7192882238861387,1.5145545446693554,-0.22203473501187787,0.355827197691827,0.8097454159042172,0.1953936026458436,0.9408493229630999,0.03377024171156151,-1.1151059457910506,-0.7917525780163359,0.9141859333905877,-1.03369509188882,-1.0799908066697839,0.6928073530013457,-1.5067422449460466,-0.19297113969960478,-2.072203528363037,0.7675250334474804,0.5068116498952335,0.04171127808145769,0.9587663595891804,0.7113753317342209,1.1141175313906793,0.23497391105626875,-0.67193504103642,-0.8605220991382723,0.790564170809116,1.3309296031501994,-0.47916951142594755,-0.3090181493198507,1.0900918736465037,-0.29118366380407323,0.5188570831919925,-0.09007009571098124,0.5219522560221179,0.15526682636179956,-0.12340008689717408,0.5058718972546394,-0.2810913542260707,-0.48133992567294115,0.9992625470700567,0.20339100462125614,-0.9042824048620446,-0.9888926573745849,0.6852001568028676,-0.5345346892794645,-0.3512343733672374,-0.48935865430639325,0.19684147893261894,-0.27244377259592056,-1.0221718578237393,-1.4637860316635276,1.0597786029665102,-0.5880712478947281,-0.15790410696201462,-0.02278151814323337,-0.6711092281917352,-0.8247833431885176,-0.04485384518693181,2.199161939031935,-1.5047975419205883,-0.019495673250080614,-0.6825787727313681,0.549500837762551,0.13656262256596108,-1.9183329320195952,-0.4951921491384681,-1.0763631811804508,-1.630358295715073,-2.584045547603697,3.283640768682705,0.03273811368895707,-1.809497715009556,-0.5824440573184653,3.4153361940463256,0.5865740436143518,-0.08235907278777432,0.16822134754326257,-2.405444958649827,-0.033713865642059915,1.1035028318764795,1.373558659326779,2.174013287467247,-0.5739155318599032,0.4983253366856566,-0.11418560646863085,0.384766552805349,-0.022342130296237883,-1.5902322552713066,-0.6506713915191753,-2.470013722328268,-1.0834607212038883,2.3181365408181165,0.49482262963254087,-1.2664390693394978,-0.33576002254100956,-1.8522173528102925,0.35167573808339014,-1.8721674847908432,1.4571645085895923,0.5351664009262791,-0.9593904117532581,0.43615038082194246,-2.0310311646428936,0.021293524276178033,0.7378255712999326,3.818333693321474,-1.838767202402103,-0.5870059616613688,1.9415054517721408,0.7497925106820347,-0.6556658329413878,-0.8194382946092205,-1.4113896069974707,-2.1696411811092275,0.8038303660213408,-1.304264836055058,-3.1796127294394516,0.7969797342973095,2.4094350867642333,0.7050389262533763,1.0385518368947189,0.39802143086582475,-1.0502972420797008,-2.8432333931281306,4.888407014459428,-1.0608546554820464,-2.659283788172484,-0.1919653229369267,0.38049416522216717,0.9540727877321409,1.026483578240267,-0.6252376005113192,0.6124300690786733,0.21233589331970393,1.3819390776624196,-0.4036364371038381,0.02681013120469089,1.2947319316010564,0.2862720709771338,-0.4989365854791039,-0.18876295296248458,0.6781700222170173,0.6983436040797208,-0.5635564294486528,0.5446274104173199,-1.762846190760392,0.3580241821723007,-0.15148688855477488,-0.844369043478288,-0.6254171145739562,-1.3425265565742528,1.1757895058350343,-0.8689772761012016,-0.21861951519790424,0.8090730622683395,-1.9086287046288675,-1.7508954080203587,-1.8680858895005592,-0.19537375553184688,-0.557092480229926,1.7338769147118072,-0.7689396123972791,-1.4412471718632247,0.10398635783275267,-0.569598164661933,0.5841032076243202,0.9931121164442835,0.6431630035130809,0.22559505769033095,-2.507572509428968,-4.414078314103254,1.4719630141764917,-2.618189289010875,0.5854125809959143,-1.9819183398859685,1.7473136381621401,-2.4372000642501668,-1.8086276850114658,-0.40719959280277607,1.9713495937858205,1.1219200241235683,2.525347977052128,-2.4733008631351083,-0.4641061000516089,1.0726709241983379,-1.2540240050005202,-3.4998474863695783,-0.29053254178794674,4.275291256457159,-2.0185947768790236,0.8084625741151541,2.6706218534382486,-0.21334215177297722,1.499139325079657,-0.3232888937573897,1.02299484353221,3.8153224612819336,1.246472157459877,-0.8579449452890131,-1.509345996239908,-1.223602309676064,1.5815169334221224,0.4684854284302098,1.4898371688669698,-0.28691916903680337,1.3498341333690433,0.07815026440551875,-1.4984400402496156,1.4337158041675124,-0.24504581263649072,-0.18299604155211924,-0.4497819849754947,-0.028675837140923477,-0.7605458518434638,0.2605639984235353,0.041619062955433606,2.805552472250117,0.2720053923210841,0.2689432585347262,1.602466399468927,-0.26313323813935696,-1.564819557868713,0.31910513803694585,-1.188631035555332,-0.7824599968594192,-1.577935683025622,0.6182367471436301,0.3854657896242841,0.5095344420171344,1.2906511152814382,-1.0604671070517213,0.5441677531422329,-0.9441752309107265,-0.8391059693710012,0.11632044504512606,-1.1411763934989771,-1.765655952524181,-0.4527114903521756,-0.4895135563377728,0.3428243623355698,0.8671050277269998,1.1300125686503293,-0.9494962824297737,-0.5523858750084543,-0.15110426061601956,-0.46873747514286634,1.2415694605658167,-0.148531887738379,1.0028092748607165,-0.18372031953726164,-1.2397781452264667,-2.3778751591062353,0.6855443144328177,-1.4502635099181482,-0.11347470904841117,2.34149842801612,0.6221636183764367,-1.0942534095117473,0.9921915269545778,-0.5557044436707237,0.6034581848588516,-0.3437981519430089,-1.6504069182109227,-1.64054710952648,1.5951946451383314,1.0395313648684734,0.23994963251192658,-0.1078624712171461,0.5555191767243173,0.3872904801506927,0.3956163451654504,-0.5256489507198481,-1.1487474297155131,0.32600773970976604,-1.748001485631523,-0.008447544238521232,0.5670733180761722,0.09490058999093165,-0.3835177629204267,-0.7366528713538045,0.3085178332600691,-1.680344274131862,0.809136487264972,-1.2153029747327793,0.6932045963517375,-0.27671178311265976,1.9463667844044963,-0.4839057015012749,-1.5123798312276726,-1.1211856151384758,-1.7150152517937833,0.4021844828029586,-0.056475570133596165,-0.10597092294304389,-2.2644739524910307,-1.8812873630357885,0.6206646856274922,-5.046795611964505,-0.5821885487279133,-1.6243708789402511,-3.1552520202460244,-1.9217743655143948,3.8242592328925795,0.8751279096154363,-0.5180040409749509,1.1423625042951357,2.168515631395973,-0.22985376834681048,1.2230430717769636,1.973126217949872,-0.24744506632758956,0.16347428377718742,-0.8379447691782893,-0.5369933636364621,2.2679118204940174,1.5093853148058223,-0.5280796705704027,-0.15310060121560554,0.6912018206392645,-1.8808418730365872,-0.7904204076551125,0.4546964385360397,1.5180282048761207,-0.47087427440999363,0.03870284602740526,1.3627707082983775,0.41003541827411283,-2.4106545567608606,-1.7314960258630663,-0.0009445311717648508,2.268876076948967,0.285541123796218,1.4283101204724424,1.442165639966575,-0.27386160453115593,0.2452618012998009,0.15811870302151476,-0.17690236064016318,1.8393206395520778,0.1859400674182332,0.09021691668014142,-0.3147866790945031,1.083993699569774,-0.7324400722220403,-0.4939282350185259,1.1637896146467597,0.6135906838379057,-0.6866362913380719,-1.8420007739690547,1.0296585159018434,-0.0657995796988331,-0.8416492369557993,1.3100837588458702,2.2072836804750473,-0.2780809994709976,-0.7559921675240666,-1.5995192813450743,-0.2784784893450798,0.02726114159079529,-1.5679765838187976,0.33088196790034274,0.9984817272879809,0.7517827590119097,0.5792882180148062,-0.8412407406419575,0.43416712402084556,-0.02096412351741961,0.41150274377335094,0.6728843944818763,0.33561405511903786,-1.1208002766782243,0.5723683414381675,0.9138210015534831,0.6193921629687793,-0.6520083779635755,-0.35948215897976254,-0.4810291511041953,0.12199162827455368,-0.9512981743034388,-0.6295914394773838,0.6810765261911378,-1.6378520307503308,0.9980500895811595,-0.997162343716429,-0.7361227553309924,0.7893285938507688,0.3512738356994838,0.09976075776174653,-0.019411914553835175,-2.903014704307516,1.1679179663278916,1.4823542771166487,-0.7103564187944222,-2.5048237653525263,-0.404669997211915,-0.5990282514576784,-0.30873036396634745,1.4749394964113094,1.4238312622986804,-0.4000961540564063,0.9892228456765295,-0.18165902910887155,-1.1077834154741062,-1.9054196831756798,1.3423279653596687,-1.5631870982089957,0.8190626490589104,1.781555331272422,-0.004511337039048158,0.04596196085939752,-0.2851221362172751,2.2381936759721555,1.9303840599389055,-0.6695692858893487,1.4405318809067147,-0.45080094677425464,0.5831590240682694,0.338678338377818,-0.2296691475939916,-0.9906654271423676,-0.09101585694812897,1.1314362322798033,-0.05232082825070705,0.4058494012949332,-1.588229004310282,0.1509464895336876,-0.4247964009510774,-1.475969773291208,0.5831686433081629,0.5334904832544396,0.2452091819326263,-0.07410653497617048,-0.8566626024345325,-1.004357014744528,0.0834858428387264,-0.32102061624392775,0.22776132641704194,-0.667830318334417,1.1270938110601314,-2.3974391934642014,0.6610462652368199,-0.7643797291026821,2.314838581733536,-0.20404622506813297,-0.14218590458829894,0.06223867089234503,-1.5531067223169437,-0.6220616184990924,-0.7128188502951716,-0.15197474373618292,-1.3986789983883696,-1.0150565984370807,1.0707956724208323,0.8819512813707853,-0.17502394888131434,-1.031524993824906,-1.9529343730772197,-0.30255131620293696,3.68177490173327,-1.4654278786014885,-1.6936513857845448,1.838005164181844,2.0923086348229702,-0.5675600101533007,0.6540958087045468,0.7583409320144341,-1.3382198273342638,-1.1051057271855846,0.5669727482430886,1.0132408725610276,-1.2160276944843673,-0.858977285550111,-0.6432645250914248,-0.2813838339227104,0.2326655466499794,-2.510778923390955,0.0009500473268531096,0.8811046392799504,-1.996170225498961,-0.15753498777506536,0.31190889171668346,-1.6768654835692232,0.6549180229590057,-1.0542916188877705,-0.22708695960007574,2.828754471220485,0.09356739157810878,-1.667412589697559,-1.3522534871660774,-1.7189663735462268,0.36127273561472606,-1.6842587137913867,-0.267736334886326,1.7422543281651492,-0.08280985606887725,-2.161974117817511,-2.145391137371888,0.6203715794777409,3.607151542392373,-1.225853362175311,-2.5029638999839676,-3.4552143879865937,0.7468239888129169,-0.6469201868393171,-0.3762288309027362,-1.868246242789251,0.5644765961232423,-2.739036784749249,-3.910728204403746,-3.3464413583518926,-2.7388765461895836,5.574430073072101,-0.564768376046576,1.6742769355142961,5.266657578663914,0.4240359447777564,0.9744307573098965,-3.001987121282479,-2.335286344255305,-3.4897957036707616,2.947973294831563,1.9812450741447662,3.219840306463073,0.3505866481754438,-0.5340908937404798,-2.1913467832301414,3.9265919934239335,1.149283142488855,-0.15922155724439335,2.3337416332974925,-1.4056487064282308,1.2645976633785416,1.395379169209754,-2.294751457072941,0.7507436228809006,-1.394776603224468,-1.8298241929992187,-0.007627405871701853,-0.18867002763469642,-4.242339341216006,1.7365040584202518,-1.6078876892041016,1.666070780383322,1.0062013318206553,0.7304693185959721,-0.01429388170057777,-2.5096553276245714,0.3264473437284643,0.14486731439928846,-1.9925579083989398,-0.46586940127308907,-3.052586363662968,0.24515861335681066,-1.392443727565329,-1.7435555130788316,2.455893115906793,1.8045985891996352,-3.279101256187289,-1.4720341077597787,3.9242818149982206,-2.3870046027977487,-1.088137363174635,-2.268656036430894,-1.9296107204994861,1.6757189063087097,-2.76661328556731,3.4368305374912644,3.831515275427666,0.7299818869314301,4.6249359381019595,-2.4408403552564018,2.2056748691816175,0.5885742041165543,1.7185412796498767,-1.0682380058028684,0.6695475989807242,-1.5337878700174556,1.7021475245864615,-0.830043021344018,-0.9117697059544128,0.9656785418598268,-0.5572877359995232,1.7014841085771337,0.8168854538514125,0.7720479896962118,0.9102088108740247,-2.3117400497877583,-0.8801745870027133,-2.0950197423162287,0.7685622772081402,-0.047577202593007684,2.433519505805659,0.6129777075070564,-1.0463120871499143,-0.40287128669076466,1.9228744209683235,2.1224758575344893,-0.3443660309382146,0.24900540081214118,-2.1190489549923086,2.7601658391305683,-0.273040305461425,-0.19028559485541052,-1.485520481553415,-0.3448767210093351,-1.2093615480261555,0.6245956702366939,-2.242599675717738,1.386090483536595,0.05637307231891597,-0.3741969589411198,-1.3511798722969814,2.324708990792068,-0.8172621649879197,-1.3232375158207077,-0.7739264840427751,1.293270303005567,0.18068811439126975,-0.5486545110176877,-0.31535302329537274,-1.61583847435951,0.4547550176916104,0.0488846701856005,3.2660779783744154,-0.04841517974485399,2.2225536744798546,-0.6566129393664077,-1.4185718606693598,1.9200605096116874,0.0,-0.14132873090133558,0.5812928710450471,-1.453093053794241,-0.39501967589785636,-1.4116164949793841,-1.582690218926519,-1.0487202543405516,1.0171738534243246,-2.966925335312265,-1.629178392491859,1.2966029965087777,0.6756084813591912,-2.3114366741091352,1.3044896853315002,0.5384559750259754,-1.4187950009778127,-0.5449844606889798,-0.6469515869428122,-1.0144971252747048,0.7540553124076865,-0.8131297549704541,-1.2768116472504865,-0.1601415958644381,2.1208135010073903,1.9311856020542528,-0.7121386038708266,-0.9546958598562583,1.3137558359497703,-1.8551733602879317,0.7168734783189912,-1.3228554649494129,1.1689063373946595,0.46114544376738564,-0.7565880044885454,-1.5475934286189208,0.9516715279377586,-0.9382919681280022,0.8225461475194429,-2.6169212219941507,-0.8394475604124096,0.4311775539417084,3.387670798435849,0.16454731623971597,3.4813555475767366,0.4233119158243426,0.8698863635811627,0.18860066483217253,2.2716697725486235,-3.5867072005637546,1.2087773972964122,-0.5490935386195162,-0.1645115873371239,-0.5796739066517453,-1.790102842169894,1.2070491156847218,2.578077903873144,-0.5064468000460565,0.27345359903403477,0.12281435481282799,-0.1865540720373815,-1.4103460574070859,0.6434326402180446,-0.3770412759865671,1.9310212409311411,0.6283683283644592,-1.231156010963641,1.5416286320617978,-0.17585306602905248,2.147160191927888,0.8358950257412623,-0.8293119666655357,-1.39431230303515,1.767755214069311,0.3273896797263198,-0.1002904000634075,-0.06764000321251018,1.4926813356018087,-0.38026243722910635,0.8425928156407636,-0.5118030897537373,-1.010876813723005,1.4177391072205836,1.2456788912182937,-0.29776244868626733,-0.3210027281244088,0.7836890453351231,-0.2506646088822784,-2.5233601328225497,-0.1130171269440261,0.40040235509639643,0.9109164098065392,0.14476258629478878,1.9377705576913762,0.1827050484366488,-0.376271533386241,1.4582939754052227,0.3532574496315277,0.47045295632690554,1.499669753561328,0.39466398692091786,-0.24065990379806834,-1.2048908026529581,0.9088363488365836,1.275285954192995,0.1032099456004687,-0.9964276879708578,2.213810397139504,0.09097933942729952,-0.15530568226109764,-1.5353545284217063,0.09947699442571789,-1.4211308523311317,0.1842152254590168,-0.8301180442180645,1.0708159801392725,-0.9785848115296147,-0.17634221941440487,0.7992609030629261,1.1568537990297223,-0.8133605123059162,1.8813461700011302,0.3399210118090856,-0.5601870187960323,-1.3571901435387446,0.9502601021902812,0.5714271293164153,-0.3432659541341643,-0.6347400836106409,-1.2511471884522507,1.1736573876202527,-1.4725555660929146,0.946669309375725,0.050588856110334035,-0.7116032016643459,1.7250531058644825,-0.2165170238320968,-0.7291794642640959,-0.18110751094670974,0.28516633917794465,-1.0312377744337908,0.2720351920196151,-1.7821303096498653,0.1676545216931835,0.7253254503106099,0.35558926451670914,0.306407201013581,0.9903932907896262,-0.6401977906381973,0.656355823157373,0.016149264671838637,0.9113770432350954,0.26072629330755603,-0.20472761120187144,0.29647420989675766,-1.0228880927174444,0.0654352558827398,0.30344650572473597,0.006019302668835991,0.60511869155917,0.5142098412679985,1.3773597057573639,0.4179192298400025,0.16550660054290844,-0.6421496131128634,0.9985324146472685,-0.824414907561799,-1.2044843940173244,0.5425842320931018,0.2171959295181658,-0.3757191250412184,1.419693669874711,-0.32598592917558733,1.3178264817757857,-0.316001861519144,-1.3093199239063047,0.6053113875182703,-1.92803531972352,-0.6071600818958522,-0.6419680880704881,0.9713458955258325,-1.0596634636558022,2.2094147768935812,0.19819076163960503,0.9375253329093169,0.43600496399580635,0.4726112246666858,-0.5277283607316718,0.4776778293636319,-0.06451504587907664,0.6998091233543668,0.3943049734567161,-0.25950955419711264,0.3157088384552864,-1.0290175997064388,0.514406892483521,0.12914035270528146,-1.5093832463008106,0.33040923721134874,-0.47123110381805944,0.21552705940868222]},"var":[[3.06364937,3.30089899],[2.38795929,1.96823257],[1.81263354,0.61977285],[4.4203603,3.07043805],[1.41879056,2.12372405],[0.25326815,1.74466499],[0.21441055,1.07944929],[0.48489682,0.5845475],[0.80558673,-0.48679685],[1.3695909,-1.26595923],[0.64179861,0.782555],[0.56131747,0.16367497],[-0.69935714,-1.36758574],[-0.30712345,-2.99335972],[0.3229298,1.49644366],[-1.75445895,-0.79270468],[-0.93696565,-0.55138708],[-2.38508928,0.34916932],[-1.63139086,-0.52117804],[-2.33511571,-2.51498253],[0.58962801,-0.26007645],[0.00424869,-0.61412362],[-0.14886421,-0.6615977],[0.72904663,1.83397564],[-0.02705528,-0.70107567],[0.08757963,-0.97469357],[-0.2792577,-2.23536615],[-0.90264813,-0.76492076],[-1.54058108,-2.35889753],[-1.17699158,-2.41021425],[0.32654753,0.71245694],[-0.30966965,-0.89631389],[-0.2308146,0.45679282],[-2.4232638,-0.61839379],[-0.27461156,0.45029911],[-1.53343919,-1.66645727],[-1.25257614,-2.01247739],[-0.0516561,-1.26594078],[0.13409731,0.44528901],[0.83203481,0.74959519],[0.68116017,1.18236871],[-0.42870886,-0.24461534],[-0.04250149,1.04839361],[1.6181109,1.70225157],[-0.09529535,1.56574014],[-0.1881152,-0.30191038],[-1.89626233,0.54876412],[-2.26878265,-2.04777615],[-1.08191962,-2.01551182],[-1.18229086,-1.50599951],[-1.44312291,-0.5428614],[0.03761662,-0.17168883],[0.69182415,-0.2493955],[-0.52242223,-1.46210432],[-0.47302242,-1.22668508],[0.01559344,-1.13978904],[-1.51542672,-1.85749092],[0.03907778,0.63219831],[-1.25182017,-1.13301139],[-2.96578974,-2.62312393],[-0.14504898,-2.00383076],[-0.13325107,-0.4520061],[-0.89848271,-2.38537731],[-0.46543282,-1.67033953],[-0.79594274,-1.58232622],[-1.99314602,-0.67807272],[-1.01591248,-2.31721799],[-0.68700491,-2.89654494],[-0.28801977,-1.34253045],[-0.37610068,-0.56614433],[1.5556845,0.38545561],[1.34805224,-0.04487417],[1.4005017,1.37057286],[2.20746846,1.84674338],[1.59836874,0.7879527],[-0.06692165,-2.11019575],[0.10987868,-1.91772224],[-0.88686198,1.09867027],[-0.37960339,-0.09198909],[-1.24860604,1.02619258],[0.63547268,-0.51403019],[-1.01119746,-1.47713232],[0.47930648,-0.85732739],[-0.42733161,-0.14351051],[-0.61169607,-1.19816243],[-1.6936534,-2.09346088],[-0.1102632,-1.20106979],[-0.07594303,-0.39903773],[0.85577593,0.8264042],[0.55537493,0.12091968],[-0.38660729,-1.48985442],[0.75594644,-0.84562773],[0.75240312,-2.42662479],[0.50867093,-1.22639605],[0.00937475,0.31530032],[0.50231645,0.96947806],[-0.1045124,1.21292252],[-0.71113667,-1.63379859],[-1.79397286,-1.9956159],[-2.41260096,-2.94128715],[-0.8916119,-2.80089511],[0.43258666,-2.26149955],[-2.38204235,-3.19976752],[-1.6316243,-1.14791543],[-2.0650257,-1.62525255],[-1.69387009,-1.38335593],[-0.82442446,-1.66697479],[1.21825994,-0.56928567],[1.7772598,0.8236301],[1.21161675,2.50986734],[0.551361,1.8156669],[0.97495876,-0.06885534],[3.03699394,0.55741787],[1.24948176,-0.4834328],[2.21138634,0.55203436],[1.65489335,1.3815692],[2.02280838,0.65149898],[1.43553527,-0.06914202],[-0.06426842,0.89561793],[0.30807478,0.41743392],[0.63599264,0.40625202],[0.34928327,-0.37700122],[-2.18698056,-0.80700045],[-0.96231381,-1.84493055],[-1.46687673,0.2598977],[-0.86161247,-1.33625812],[-0.92790773,-2.3191579],[-2.27586181,-2.00053325],[-0.63029125,-2.37664785],[-0.18294408,-0.21871357],[-1.66647053,-0.56810207],[-0.48332946,-0.45392054],[-2.11245107,-2.56493937],[-0.09851057,-0.37675347],[-0.24578398,-1.38200536],[-0.80812814,-0.75798637],[-1.35337736,-0.65981841],[0.1457184,-0.50960032],[0.55943057,-0.84118455],[-0.40081374,-1.42180842],[-0.1171892,-0.61679579],[-0.05105813,-1.2176018],[-0.07268232,-0.83805873],[0.88629523,-0.19342411],[1.18733732,1.23551346],[1.88630712,3.28360498],[1.44313431,2.17688855],[2.34257255,1.39999897],[-0.57137699,0.39406332],[-0.73426347,-0.5934195],[-0.0188492,0.88259448],[-0.77314766,-0.20180846],[-0.69525081,-0.88947651],[-1.7197683,-0.33895067],[-0.47113179,0.63525225],[0.62306739,0.18488668],[-0.61660291,-0.10084475],[-0.17006191,1.31807557],[-0.50582132,-1.04309127],[0.80575687,-0.05277169],[-0.42508276,-0.16351296],[0.81283916,-0.37060955],[0.9540047,-1.32327378],[1.12557271,0.25598395],[0.49658974,0.92863699],[1.99418606,1.86455453],[0.97355201,1.24700006],[1.92412177,1.92261317],[2.91602971,2.63028956],[1.84712206,1.33838357],[0.10679746,0.1809247],[0.63536118,-0.67515604],[0.50914285,0.15473675],[-0.35630366,0.37082024],[-0.37957631,-0.17726553],[-2.13909985,-3.24891504],[-0.56209955,-1.33730838],[-0.26759411,-1.7394506],[-0.99500153,0.28408961],[-0.27987465,-0.03205546],[1.17317284,0.97116241],[1.79149014,0.64697628],[2.34929274,2.36466362],[-0.23990874,-1.02020264],[-0.55141927,0.08222397],[-0.84515877,-1.84200299],[1.49368607,0.36394907],[3.12131918,1.86821122],[1.29789941,0.11399535],[4.07006271,0.68467716],[3.49054313,2.57336217],[2.47101555,2.9421716],[1.05342635,0.99317658],[0.00088084,-0.72275402],[-0.04422578,-1.0288086],[0.08778537,0.52303331],[2.22368894,1.59989711],[0.5921695,0.43966396],[2.52498397,2.44331653],[1.06759568,0.87153086],[0.41290628,1.73030555],[-0.42774466,1.70432407],[0.4027108,0.71884104],[0.51491331,1.65615237],[1.21971538,-0.53347045],[0.43056837,-0.8570556],[-0.81817481,-2.37594743],[0.26165206,0.99969434],[-1.2374888,0.85690327],[-1.57616946,-1.34804066],[-1.29391635,-1.13867285],[0.66731151,0.19093334],[1.08529767,0.1284857],[0.24814522,0.61233231],[0.09667032,-0.65256884],[-0.22657784,0.05931019],[-0.96256702,-1.61894357],[-1.134265,-1.31999858],[-0.66703757,0.40864045],[0.59440533,1.32216608],[0.43493945,1.07842546],[0.93886105,1.78924196],[0.36486727,0.49349779],[-0.18677212,-0.12357814],[-0.67278533,-0.13607663],[-0.71179712,-1.33546002],[-0.03296851,0.33966857],[-1.9950594,-0.36801491],[-1.23485986,-2.13566207],[-0.86281173,-3.48565036],[1.97306706,-1.02188484],[1.80825174,2.32486645],[0.31147385,0.08908226],[-0.52326274,0.97049609],[0.35107667,0.09313907],[-0.04351721,0.88088506],[0.97434962,1.92432183],[1.0479125,-0.57480747],[2.66682159,1.20541126],[1.88043358,3.51573398],[2.28828704,2.43376901],[2.28593295,2.54807885],[1.35107527,1.80143326],[1.14656928,1.35173669],[2.23325032,2.03889208],[-0.08811483,-1.66810871],[0.20253918,-0.44133635],[1.19019581,2.12273687],[0.50666895,1.66867378],[1.01898454,1.9546161],[1.07823197,0.15587875],[-0.02513015,-1.21631229],[-0.62680823,0.34525566],[-0.65391313,0.47651942],[-1.01097588,0.11700937],[1.74857671,1.19206793],[1.28732663,3.80046981],[-0.63085363,0.03577378],[-0.87402691,-0.06723128],[-0.54024563,-1.91310443],[-2.79760521,-4.01226132],[-2.59702825,-2.65497196],[-1.21584491,-0.00962034],[-1.09960842,0.6394051],[-0.00552767,0.82407472],[-2.61484343,-1.01643891],[-1.87538457,-0.62085525],[1.08978757,0.39900981],[-1.54322238,-1.60138804],[-0.26962194,0.2364016],[0.02080625,-0.72141399],[0.10505253,-0.34305117],[0.24412115,0.27346781],[-0.95303515,0.73088574],[-0.6491779,-1.76316406],[0.3260278,0.34827959],[-0.73422766,-1.32466044],[-2.32994073,-0.61336095],[-0.85752743,-0.85226226],[-0.96405679,-0.71855208],[-0.09547781,0.66702565],[0.7326549,-0.09524003],[-0.06838865,0.28598694],[0.54375658,-0.33063178],[-0.85602858,0.86742004],[-1.61221047,-0.50503152],[0.01987906,-1.95876385],[-0.02615382,-0.14749945],[0.4642964,-0.13730741],[0.01989689,0.44111777],[0.09380714,-0.8323183],[-0.96148804,0.84745088],[-0.96415402,0.22623784],[0.34293046,-2.03280607],[1.0152971,-1.26876817],[0.5054658,1.44967543],[0.51630391,-0.27021539],[-1.24333849,-0.06019279],[-2.11457719,-2.30241545],[-0.81728779,-2.52682516],[0.42027792,-2.76629219],[0.01296529,1.35739639],[0.51863937,0.31149055],[-0.29474219,-0.87676414],[1.8586418,0.57176039],[2.87334727,0.16954258],[3.39302336,1.19559698],[2.90470726,-0.00590323],[2.05408591,0.4230897],[1.35362938,0.44382116],[2.5327529,2.1565403],[0.79969753,1.66423897],[2.57607172,1.38128804],[-0.36117641,-1.7836649],[1.17401974,-0.34148228],[0.73759865,0.12193801],[1.9259917,2.09413423],[0.98921936,0.78956755],[2.04692174,0.49040919],[2.78212097,2.38042201],[2.59761611,1.43039437],[3.34081934,3.32011434],[5.17858405,4.19620414],[1.98036681,2.51608436],[-0.32845064,2.98918433],[0.70503584,1.11721566],[1.05406207,1.25956005],[-0.6194604,-2.60044326],[-2.08461032,-2.48053552],[-0.59106486,-1.4827169],[2.84951803,-2.19313168],[2.17500356,1.81106674],[2.69373344,2.57813754],[2.30640954,1.42277658],[0.43156134,1.19280762],[0.14765797,0.94301698],[1.00079698,2.40080903],[-0.34902024,-0.37849702],[-0.8137742,-1.68337888],[-1.98494811,-0.85024343],[-1.82212539,-0.55375524],[0.70434488,1.01421033],[1.98045958,1.61581177],[1.8154187,0.6993503],[-0.09387454,0.24644733],[3.22041516,1.90365255],[-0.29877848,-0.45228435],[-2.70418756,-3.13869554],[-2.54376855,-2.0101842],[-0.22211916,-0.67095227],[-0.78381183,-1.14677384],[-0.40568605,-0.02038781],[1.15126298,-0.52781677],[0.28590098,-0.82906379],[0.47499871,-0.14274375],[-0.48378566,-0.62781751],[0.12374398,0.26730144],[-0.8186888,-0.60319968],[0.26577891,-0.69808909],[-0.68311601,0.25839129]]}'
EXPECTED_HASH = 'b42393cc809c3a9ab4ea777013f9931f5eb121c1fae7dd23c866336ef448ee91'
assert hashlib.sha256(DATA_TEXT.encode()).hexdigest()==EXPECTED_HASH
raw=json.loads(DATA_TEXT)
returns=pd.Series(raw['garch']['return'], index=pd.to_datetime(raw['garch']['date']),name='return_pct')
signals=pd.DataFrame(raw['var'],columns=['x','y'],index=pd.RangeIndex(1,361,name='t'))
gtrain=returns.iloc[:1200].copy()
vtrain=signals.iloc[:240].copy()
print('GARCH train:',gtrain.index[0].date(),'to',gtrain.index[-1].date(),len(gtrain))
print('VAR train: 1–240; validation: 241–300; final: 301–360')
print('Data SHA256:',EXPECTED_HASH)


## Q1 · 퍼짐을 확인 / First inspect the spread
(a) Read Q1_READ. Compare the lag-1 ACF of returns and squared returns. Then use the two supplied residual p-values to say whether mean or variance patterns remain detectable. Two sentences are enough. [2]
(b) Compare the next-day and day-10 variance forecasts with the long-run variance. State which direction the forecast moves and what the long-run value means. Units are squared percentage points, not returns. Two sentences. [2]

먼저 Q1_READ 두 표를 읽고 Code 1–3을 완성하세요. 답은 (a) 두 문장, (b) 두 문장이면 충분합니다. / Read the two Q1_READ tables, fill Code 1–3, and write four sentences. Then move to Q2.

GARCH: 1,200 training returns and 10 final days. Constant mean, Gaussian GARCH(1,1) fixed in advance. Do not change the period or model to improve the final comparison.

Definitions: u=r−mu; z=u/sigma; next variance=omega+alpha*u²+beta*previous variance. Long-run v=omega/(1−alpha−beta) when alpha+beta<1. sqrt(v) is not the mean daily sigma. Provided fits remain visible if your code is incomplete, allowing supported partial credit.

In [ ]:
#@title Q1 제공 분석 / Provided analysis
gfit=arch_model(gtrain,mean='Constant',vol='GARCH',p=1,q=1,dist='normal',rescale=False).fit(disp='off')
gp=gfit.params; persistence=float(gp['alpha[1]']+gp['beta[1]'])
long_run=float(gp['omega']/(1-persistence)) if persistence<1 else None
gf=np.asarray(gfit.forecast(horizon=10).variance.iloc[-1])
gz=np.asarray(gfit.std_resid)
table('GARCH estimates',pd.DataFrame({'estimate':gp,'robust_SE':gfit.std_err}))
RESULTS['GARCH summary']={'persistence':persistence,'long_run_variance':long_run,'sqrt_long_run':None if long_run is None else float(np.sqrt(long_run)), 'variance_h1':float(gf[0]),'variance_h10':float(gf[-1]),'convergence_flag':int(gfit.convergence_flag)}
if SHOW_DETAILS: print(RESULTS['GARCH summary'])
if gfit.convergence_flag: print('적합 경고: 수렴 실패. 해석에 한계를 기록하세요 / convergence warning')
if SHOW_DETAILS:
    gtrain.plot(color='#285943',title='Training log returns (%)');plt.tight_layout();plt.show()
acf_panel([gtrain,gtrain**2],['Returns','Squared returns'])
if SHOW_DETAILS: acf_panel([gz,gz**2],['Standardized residual z','Squared standardized residual z²'])
gd=[]
for name,v in [('r',gtrain),('r2',gtrain**2),('z',gz),('z2',gz**2)]:
    lb=acorr_ljungbox(v,lags=[10],return_df=True).iloc[0]
    gd.append([name,*acf(v,nlags=5,fft=False)[[1,5]],float(lb.lb_stat),float(lb.lb_pvalue)])
table('GARCH diagnostics',pd.DataFrame(gd,columns=['series','acf1','acf5','LB10','p']))
table('Q1_READ diagnostics',pd.DataFrame(gd,columns=['series','acf1','acf5','LB10','p'])[['series','acf1','p']])
table('Q1_READ variance',pd.DataFrame({'item':['next day','day 10','long-run'], 'variance_pct_squared':[gf[0],gf[-1],long_run]}))
print('Read r/r2 ACF and z/z2 p-values. p>=.05 means no pattern detected, not proof of independence. Long-run variance is the finite unconditional variance under this constant-mean model; it is not tomorrow’s return. It requires persistence<1. No derivation required.')
print('Ljung–Box values are approximate diagnostic evidence after estimation, not proof of independence or normality. Reference ACF bands are pointwise and approximate.')


### Code 1 · standardize
Subtract the predicted mean, then divide by the conditional STANDARD DEVIATION. Week 9 slides. r, sigma: same-length arrays; mu: scalar; return z array. / 평균을 빼고 표준편차로 나누세요.

`None`/`np.nan` 부분만 채우세요. 미완성·수치 오류는 점검에서 안내하며 뒤 분석은 계속됩니다. SyntaxError는 수정해야 합니다. / Fill the blank lines; no solution cells are included in the student version.

In [ ]:
def standardize(r, mu, sigma):
    u = r - mu # provided
    z = None # TODO 1: divide the shock by sigma
    return z
check(1,standardize,[( (np.array([1.2,-.8]),.2,np.array([.5,2.])), np.array([2.,-.5]) ), ( (np.array([.4,.8]),.0,np.array([2.,.4])), np.array([.2,2.]) )])

### Code 2 · next_variance
Use the unstandardized shock u, its square and the previous VARIANCE. Week 9 slides. All arguments scalar; return scalar variance. / 표준화 잔차 z를 대신 넣지 마세요.

`None`/`np.nan` 부분만 채우세요. 미완성·수치 오류는 점검에서 안내하며 뒤 분석은 계속됩니다. SyntaxError는 수정해야 합니다. / Fill the blank lines; no solution cells are included in the student version.

In [ ]:
def next_variance(omega, alpha, beta, u, previous_variance):
    value = None
    return value
check(2,next_variance,[((.02,.1,.85,-2.,1.),1.27),((.02,.1,.85,2.,1.),1.27),((.10,.2,.5,.5,2.),1.15)])

### Code 3 · forecast_variance
For h>=1 and persistence<1, forecast the excess above long-run variance. h=1 is tomorrow. Week 9 slides and lab. All scalar; return variance. / h−1 지수를 확인하세요.

`None`/`np.nan` 부분만 채우세요. 미완성·수치 오류는 점검에서 안내하며 뒤 분석은 계속됩니다. SyntaxError는 수정해야 합니다. / Fill the blank lines; no solution cells are included in the student version.

In [ ]:
def forecast_variance(h, first_variance, long_run, persistence):
    excess = first_variance - long_run # provided
    value = None # TODO 3: long-run plus the remaining excess
    return value
check(3,forecast_variance,[((1,1.77,.4,.95),1.77),((2,1.77,.4,.95),1.7015),((3,.2,1.,.5),.8)])

In [ ]:
# 학생 코드와 실제 적합 연결 / Connect your code to the actual fit
if CHECKS.get(1):
    assert np.allclose(standardize(gtrain.to_numpy(),float(gp['mu']),np.asarray(gfit.conditional_volatility)),gz)
if CHECKS.get(2):
    assert np.isclose(next_variance(gp['omega'],gp['alpha[1]'],gp['beta[1]'],float(gfit.resid.iloc[-1]),float(gfit.conditional_volatility.iloc[-1]**2)),gf[0])
if CHECKS.get(3) and long_run is not None:
    student_gf=np.array([forecast_variance(h,gf[0],long_run,persistence) for h in range(1,11)])
    assert np.allclose(student_gf,gf)
    table('Student variance forecasts',pd.DataFrame({'h':range(1,11),'variance':student_gf,'sqrt_variance':np.sqrt(student_gf)}))


## Q2 · 다음 값 예측의 후보 확인 / Check a forecasting candidate
(a) In Q2_READ, check VAR1 stable and residual p-value. State whether these checks give a reason to stop using this candidate, with one limitation. One sentence. Stationarity preprocessing is provided; do not repeat ADF/KPSS analysis. [1]
(b) Read the x-to-y Granger p-value for VAR1. Does it support including x past in the y equation? Explain why this does not prove that changing x causes y to change. Two sentences. No F or degrees-of-freedom calculation or transcription is required. [2]

Q1은 수익률의 퍼짐을 다뤘습니다. 여기서부터는 별도 모의자료의 y 다음 값을 예측합니다. x를 함께 쓰는 VAR가 후보입니다. 두 자료를 합치지 않습니다. Code 4는 지난 두 값을 다음 예측으로 바꾸는 한 줄입니다. 읽을 출력은 Q2_READ 하나입니다. stable과 두 p값만 보고 짧게 답하면 다음으로 갑니다. / Read Q2_READ only; stationarity preprocessing is supplied. Advanced tables are optional.

In [ ]:
#@title Q2 제공 분석 / Provided analysis. Training only.
if SHOW_DETAILS:
    vtrain.plot(subplots=True,title='Training monitoring signals (index units)');plt.tight_layout();plt.show()
st=[]
for c in ['x','y']:
    with warnings.catch_warnings(record=True):
        ap=adfuller(vtrain[c],regression='c',autolag='AIC')[1]
        kp=kpss(vtrain[c],regression='c',nlags='auto')[1]
    st.append([c,ap,kp])
table('VAR stationarity',pd.DataFrame(st,columns=['signal','ADF_p','KPSS_p']))
if SHOW_DETAILS: print('ADF H0: unit root. KPSS H0: level stationarity; .01/.10 may be table bounds. No automatic guarantee.')
VF={p:VAR(vtrain).fit(p,trend='c') for p in [1,2]}
vd=[];gc=[]
for p,f in VF.items():
    w=f.test_whiteness(nlags=10,adjusted=True)
    vd.append([p,f.aic,f.bic,f.is_stable(),float(np.min(np.abs(f.roots))),w.test_statistic,w.df,w.pvalue])
    for target,cause in [('x','y'),('y','x')]:
        test=f.test_causality(caused=target,causing=[cause],kind='f')
        gc.append([p,cause+' -> '+target,test.test_statistic,str(test.df),test.pvalue,'reject' if test.pvalue<.05 else 'do not reject'])
table('VAR diagnostics',pd.DataFrame(vd,columns=['p','AIC','BIC','stable','min_abs_roots','white_stat','white_df','white_p']))
table('Granger tests',pd.DataFrame(gc,columns=['p','direction','F','df','p_value','at_5pct']))
if SHOW_DETAILS: print('Different lag orders use different start rows; this is exploratory sensitivity, not pure controlled lag inference. statsmodels system F degrees of freedom are reported.')
if SHOW_DETAILS: acf_panel([VF[1].resid.x,VF[1].resid.y],['VAR(1) residual x','VAR(1) residual y'])
# Show lagged cross-correlation in addition to marginal residual ACFs.
u=VF[1].resid.to_numpy()
cross=pd.DataFrame({'lag':range(1,6),'x_t_y_past':[np.corrcoef(u[h:,0],u[:-h,1])[0,1] for h in range(1,6)],'y_t_x_past':[np.corrcoef(u[h:,1],u[:-h,0])[0,1] for h in range(1,6)]})
table('Residual cross correlations',cross)
table('VAR1 coefficients',VF[1].params)
q2test=VF[1].test_causality(caused='y',causing=['x'],kind='f')
table('Q2_READ',pd.DataFrame([{'model':'VAR1','stable':VF[1].is_stable(),'residual_p':vd[0][-1],'x_to_y_Granger_p':q2test.pvalue}]))
print('Preprocessing is provided: the training series were checked for stationarity. stable=True means shocks fade. Residual p>=.05 means no lagged pattern detected. Granger p<.05 is evidence against zero x-lag coefficients in the y equation, not intervention causation. No need to transcribe the optional tables.')


### Code 4 · var_one_step
Row=target equation, column=whose past. Use @ for matrix multiplication. Week 10 slides. intercept,last: (2,); matrix:(2,2); return (2,) prediction. / 각 행이 한 방정식입니다.

`None`/`np.nan` 부분만 채우세요. 미완성·수치 오류는 점검에서 안내하며 뒤 분석은 계속됩니다. SyntaxError는 수정해야 합니다. / Fill the blank lines; no solution cells are included in the student version.

In [ ]:
def var_one_step(intercept, matrix, last):
    prediction = None
    return prediction
check(4,var_one_step,[((np.array([0.,0.]),np.array([[.5,.2],[.1,.4]]),np.array([2.,1.])),np.array([1.2,.6])),((np.array([1.,-1.]),np.array([[.2,.3],[.6,.1]]),np.array([1.,2.])),np.array([1.8,-.2]))])

## Q3 · 반응 해석에 필요한 가정 / Check the assumption
Compare only the same-time (h=0) response of x to the standardized y shock under the two orderings. Report both values and explain why the forced zero is an ordering assumption, not evidence of no effect. The shock definition and scale change with ordering; these curves alone do not establish causation. Two sentences. [2.5]

Q2에서 후보를 확인했지만 아직 인과를 말할 수 없습니다. Code 5의 반복문은 제공되어 있으므로 안쪽 한 줄만 채우세요. 읽을 출력은 Q3_READ same-time ordering 하나이고 두 문장이면 됩니다. Code 4–5 의 장난감 출력은 코드 연습이고 채점 문항이 아닙니다. / Read only Q3_READ same-time ordering. The toy output under Code 4–5 is coding practice, not a graded question. No hand matrix derivation or extra horizon table is required.

### Code 5 · impulse_path
Initialize the h=0 shock (provided), then propagate the path DIFFERENCE. Do not add an intercept. Week 10 slides. matrix:(2,2), shock:(2,), horizon:int>=0; return (horizon+1,2). Replace the np.nan line. / 수준 예측이 아니라 경로 차이입니다.

`None`/`np.nan` 부분만 채우세요. 미완성·수치 오류는 점검에서 안내하며 뒤 분석은 계속됩니다. SyntaxError는 수정해야 합니다. / Fill the blank lines; no solution cells are included in the student version.

In [ ]:
def impulse_path(matrix, shock, horizon):
    path = np.empty((horizon+1,2))
    path[0] = shock
    for h in range(1,horizon+1):
        path[h] = np.nan
    return path
check(5,impulse_path,[((np.array([[.5,.2],[.1,.4]]),np.array([1.,0.]),2),np.array([[1.,0.],[.5,.1],[.27,.09]])),((np.array([[.2,0.],[0.,.3]]),np.array([0.,2.]),1),np.array([[0.,2.],[0.,.6]]))])

In [ ]:
toy=np.array([[.5,.2],[.1,.4]])
toy_path=np.stack([np.linalg.matrix_power(toy,h)@np.array([1.,0.]) for h in range(3)])  # provided reference / 기준 계산
RESULTS['toy_prediction']=[round(float(v),4) for v in toy@np.array([2.,1.])]
RESULTS['toy_h1_y']=round(float(toy_path[1,1]),4)
print('Practice toy for Code 4–5, not a graded question: next-value prediction',RESULTS['toy_prediction'],'; y response after unit x shock at h=1:',RESULTS['toy_h1_y'])
if CHECKS.get(4) and CHECKS.get(5):
    table('Student toy results',pd.DataFrame(impulse_path(toy,np.array([1.,0.]),2),columns=['x_response','y_response']))
    assert np.allclose(impulse_path(toy,np.array([1.,0.]),2),toy_path)
    assert np.allclose(var_one_step(VF[1].intercept,VF[1].coefs[0],vtrain.iloc[-1].to_numpy()),VF[1].forecast(vtrain.to_numpy()[-1:],1)[0])

In [ ]:
#@title Q3 순서 민감도 / Ordering sensitivity. Same VAR(1), no refitting.
f=VF[1]; irf_rows=[]
for order,label in [([0,1],'x first'),([1,0],'y first')]:
    mat=f.coefs[0][np.ix_(order,order)]
    cov=np.asarray(f.sigma_u)[np.ix_(order,order)]
    chol=np.linalg.cholesky(cov)
    response_idx=order.index(0); shock_idx=order.index(1)
    path=np.stack([np.linalg.matrix_power(mat,h)@chol[:,shock_idx] for h in range(9)])
    for h in [0,1,4]:irf_rows.append([label,h,path[h,response_idx],path[0,shock_idx]])
    plt.plot(range(9),path[:,response_idx],marker='o',label=label)
plt.title('x response to each ordering’s standardized y shock');plt.xlabel('Horizon');plt.ylabel('x index units');plt.legend();plt.tight_layout();plt.show()
table('Ordering comparison',pd.DataFrame(irf_rows,columns=['order','h','x_response','y_impact_scale']))
table('Q3_READ same-time ordering',pd.DataFrame(irf_rows,columns=['order','h','x_response','y_impact_scale']).query('h == 0'))
print('Point estimates only. Standardized shock variance=1. The definition and scale change with ordering. No confidence intervals or causal identification are established.')


## Q4 · 미사용 관측값에서 확인 / Evaluate on later observations
(a) BEFORE opening final data, use Q4_SELECT to choose VAR1 or VAR2 for predicting y. Compare its validation RMSE with AR1 and record one reason in the choice box. Use Q2 as a diagnostic check; a small Granger p-value is not the selection rule. Two sentences. [1.5]
(b) AFTER opening final data, use Q4_FINAL to compare your selected model with AR1 for y using RMSE. Keep your original choice. State what this adds beyond the Granger test. Finally, compare the GARCH forecast and observed squared shocks in the provided plot: why cannot one realized squared shock be treated as the true conditional variance? Three sentences. [2]

Q2·Q3의 한계를 기억하며 y 예측 후보를 고릅니다. 먼저 Q4_SELECT를 읽고 (a)를 작성한 뒤 공개하세요. 공개 후 Q4_FINAL과 GARCH 그림을 읽고 (b)를 작성합니다. Code 6도 이때 채웁니다. (a) 두 문장, (b) 세 문장이면 충분합니다. / Choose with validation first; then inspect final evidence without retuning.

The VAR is refitted before EACH one-step forecast, using only observations already seen. During final evaluation, earlier final observations may enter later fits, but current/future outcomes never do. Keep the chosen lag fixed throughout. The GARCH forecast is different: all ten horizons are forecast once at the training endpoint, with no future-return updates.

먼저 reveal_holdout=False로 실행하고 검증 결과를 읽으세요. 선택·근거·학번을 입력한 다음 True로 바꿔 모두 실행하세요. Code 1–5 통과는 공개 조건이 아닙니다. / Code 1–5 do not need to pass before you reveal. 공개 후 설정·선택·근거를 바꾸지 마세요. Code 6과 AI 사용란은 이후 완성해도 됩니다.

In [ ]:
#@title Q4 검증 구간 / Validation. All forecasts use past observations only.
def rolling(frame,start,stop):
    predictions={k:[] for k in ['VAR1','VAR2','AR1','Mean']}
    for i in range(start,stop):
        past=frame.iloc[:i]
        AUDIT.append((i+1,int(past.index[-1])))
        assert past.index[-1] < frame.index[i]
        for p in [1,2]:
            f=VAR(past).fit(p,trend='c')
            predictions['VAR'+str(p)].append(f.forecast(past.to_numpy()[-p:],1)[0])
        predictions['AR1'].append([float(np.asarray(AutoReg(past[c].to_numpy(),lags=1,trend='c').fit().predict(start=i,end=i))[0]) for c in frame.columns])
        predictions['Mean'].append(past.mean().to_numpy())
    return {k:np.asarray(v) for k,v in predictions.items()}
def metric_table(actual,preds):
    rows=[]
    for model,pred in preds.items():
        for j,c in enumerate(['x','y']):
            e=actual[:,j]-pred[:,j]
            rows.append([model,c,float(np.abs(e).mean()),float(np.sqrt(np.mean(e**2)))])
    return pd.DataFrame(rows,columns=['model','signal','MAE','RMSE'])
VALID=rolling(signals.iloc[:300],240,300)
table('Validation errors',metric_table(signals.iloc[240:300].to_numpy(),VALID))
table('Q4_SELECT y only',metric_table(signals.iloc[240:300].to_numpy(),VALID).query("signal == 'y' and model != 'Mean'") [['model','RMSE']])
print('Validation is available for selection, so it is not an independent final test.')
print('Q4 는 y 의 검증 RMSE 만 비교합니다. x 결과는 선택이며 채점하지 않습니다. / For Q4, compare the validation RMSE for y only. Results for x are optional and are not graded.')


In [ ]:
#@title 잠정 선택 / Commit before reveal
preferred = "undecided" #@param ["undecided","VAR1","VAR2"]
choice_reason = "" #@param {type:"string"}
reveal_holdout = False #@param {type:"boolean"}
student_id = "" #@param {type:"string"}
commit={'data_sha256':EXPECTED_HASH,'preferred':preferred,'reason':choice_reason.strip(),'student_id':student_id.strip()}
commit_hash=digest(commit)
lockpath=Path('A3_commitment_'+EXPECTED_HASH[:10]+'.json')
lock=None
if lockpath.exists(): lock=json.loads(lockpath.read_text(encoding='utf8'))
commit_matches=lock is None or lock['commit_hash']==commit_hash
eligible=preferred in ['VAR1','VAR2'] and bool(choice_reason.strip()) and bool(student_id.strip())
OPENED=False;HOLD=None
if not commit_matches:
    print('선택이 공개 당시 기록과 다릅니다 / COMMITMENT CHANGED. Restore the original choice/reason/id. Do not reset the record to retune.')
elif not eligible or not reveal_holdout:
    print('아직 공개하지 않음 / CLOSED. VAR1·VAR2 선택, 이유, 학번을 넣고 reveal_holdout=True 로 모두 실행하세요. Code 1–5 통과는 필요 없습니다.')
    print('Choose VAR1 or VAR2, enter a reason and your student ID, set reveal_holdout=True, then Run all. Code 1–5 do not need to pass.')
else:
    if lock is None:
        lock={'commit_hash':commit_hash,'commit':commit,'opened_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
        lockpath.write_text(json.dumps(lock,ensure_ascii=False,indent=2),encoding='utf8')
    OPENED=True
    FINAL=rolling(signals,300,360)
    HOLD=metric_table(signals.iloc[300:360].to_numpy(),FINAL)
    table('Final errors',HOLD)
    table('Q4_FINAL y only',HOLD[(HOLD.signal=='y') & HOLD.model.isin([preferred,'AR1'])][['model','RMSE']])
    gactual=returns.iloc[1200:1210].to_numpy()
    gcomparison=pd.DataFrame({'h':range(1,11),'forecast_variance':gf,'observed_squared_shock':(gactual-float(gp['mu']))**2})
    table('GARCH future comparison',gcomparison)
    plt.plot(range(1,11),gf,marker='o',label='Variance forecast at fixed origin')
    plt.plot(range(1,11),gcomparison.observed_squared_shock,marker='x',label='Observed squared shock')
    plt.xlabel('Days ahead');plt.ylabel('Squared percentage points');plt.legend();plt.tight_layout();plt.show()
    print('Commitment:',json.dumps(lock,ensure_ascii=False))
print('The local lock is an accidental-change guard, not a secure blind test. A new Colab runtime can lose it. Keep the original choice and disclose any earlier viewing.')


### Code 6 · error_metrics
Calculate MAE and RMSE for one signal. Week 10 slides and A2 Code 5. actual,predicted:(n,); return [MAE,RMSE]. / 두 오차의 단위는 원자료와 같습니다.

`None`/`np.nan` 부분만 채우세요. 미완성·수치 오류는 점검에서 안내하며 뒤 분석은 계속됩니다. SyntaxError는 수정해야 합니다. / Fill the blank lines; no solution cells are included in the student version.

In [ ]:
def error_metrics(actual, predicted):
    error = actual - predicted # provided
    mae = np.mean(np.abs(error)) # provided
    rmse = None # TODO 6: square, mean, then square root
    return np.array([mae,rmse],dtype=float)
if OPENED:
    check(6,error_metrics,[((np.array([1.,4.,-1.]),np.array([0.,2.,1.])),np.array([5/3, np.sqrt(3.)])),((np.array([0.,2.]),np.array([0.,2.])),np.array([0.,0.]))])
else:
    CHECKS[6]=False
    CHECK_SIGNATURES[6]=hashlib.sha256(error_metrics.__code__.co_code+repr(error_metrics.__code__.co_consts).encode()).hexdigest()[:16]
    print('Code 6: final observations must be opened first / 최종 공개 후 점검')

In [ ]:
if OPENED and CHECKS.get(6):
    rows=[]
    for model in [preferred,'AR1']:
        for j,name in enumerate(['x','y']):
            got=error_metrics(signals.iloc[300:360,j].to_numpy(),FINAL[model][:,j])
            ref=HOLD[(HOLD.model==model)&(HOLD.signal==name)][['MAE','RMSE']].to_numpy()[0]
            assert np.allclose(got,ref)
            rows.append([model,name,*got])
    table('Student final metrics',pd.DataFrame(rows,columns=['model','signal','MAE','RMSE']))
else: print('Code 6 report pending; submit available evidence even if incomplete.')


## Q5 · 숨은 수준 따라가기 / Track a hidden level (Week 11)
이 문항은 11주차(11월 10일) 칼만 필터 수업 내용을 씁니다. 그 수업 뒤에 하세요. 코드는 쓰지 않습니다. / This part uses the Week 11 Kalman filter class on 10 November. Do it after that class. No coding.

(a) Read Q5_READ gain. The gain K starts large and then settles. Why does it settle, and what does the settled value say about how far the filter moves toward each new reading? One or two sentences. [1]
(b) Read Q5_READ variance around t = 60–69, where no reading arrives. What happens to the variance P during the gap and right after readings return, and why? One or two sentences. [1]

자료는 앞 문항과 별개인 모의 센서입니다. 숨은 수준은 매 시점 조금씩 움직이고(그 분산 Q = 0.25), 센서는 수준에 잡음을 더해 읽습니다(그 분산 H = 2). 필터는 두 걸음을 되풀이합니다. 예측: 수준이 움직였을 수 있으니 P 에 Q 를 더합니다. 갱신: 예측에서 읽기 쪽으로 이득 K = 예측 P / (예측 P + H) 만큼 옮기고, P 를 (1 − K) 배로 줄입니다. 읽기가 없으면 갱신을 건너뜁니다. / A separate simulated sensor. The hidden level moves a little each step (variance Q = 0.25), and each reading adds noise (variance H = 2). Predict: add Q to P. Update: move from the prediction toward the reading by the gain K = predicted P / (predicted P + H), then multiply P by (1 − K). With no reading, skip the update.

In [ ]:
#@title Q5 제공 분석 / Provided analysis. Week 11 Kalman filter, no student code
LEVEL = [-0.007,0.131,0.242,0.180,0.655,-0.605,-0.625,-0.639,-0.816,-0.646,-0.178,-0.471,-0.747,-0.607,0.108,0.869,1.087,1.218,1.545,1.367,1.344,1.130,1.849,1.581,1.323,0.590,0.931,0.303,-1.314,-0.538,-0.429,-1.350,-1.949,-2.203,-1.519,-1.168,-0.917,-1.107,-1.164,-1.801,-2.293,-1.882,-2.026,-2.052,-1.577,-0.567,-0.731,-0.718,-0.368,-0.544,-0.764,-1.665,-1.661,-2.080,-1.703,-1.888,-2.071,-1.216,-1.748,-2.299,-2.129,-3.047,-2.948,-3.123,-2.144,-1.750,-1.803,-1.617,-1.275,-1.797,-2.222,-1.642,-1.458,-0.866,-0.560,-0.330,-0.419,0.528,0.602,-0.021,-0.240,0.079,0.624,1.057,1.989,1.955,1.687,1.724,2.206,2.682,3.246,2.964,2.490,2.430,2.399,1.234,1.976,2.176,2.263,2.428]   # hidden level, t = 1..100 / 숨은 수준
READING = [0.424,-3.838,-1.252,1.127,0.714,0.746,2.262,-1.785,-0.364,1.164,0.192,-0.511,0.710,0.219,-1.186,1.683,0.928,-0.188,1.225,1.343,5.045,0.640,1.138,2.578,1.108,1.058,-0.262,-1.784,-1.425,0.440,2.070,-0.313,-0.492,-2.346,-4.277,-2.340,-0.845,1.303,-3.409,-3.861,-3.672,-1.108,-1.620,-3.252,-1.147,0.944,-1.624,-0.960,-1.745,-1.302,-1.839,0.321,-1.951,-1.886,-0.391,2.964,-1.421,0.543,-1.980,None,None,None,None,None,None,None,None,None,None,-2.374,-1.594,-0.096,-0.774,-2.962,0.611,-0.474,-0.912,-0.513,1.131,-0.016,0.117,-0.750,0.794,0.395,-1.160,0.143,1.101,2.860,5.120,3.156,2.324,2.825,2.314,2.936,1.194,3.382,1.111,1.832,1.480,2.868]   # sensor reading = level + noise; None = no reading / 센서 읽기, None 은 읽기 없음
Qv, Hv = 0.25, 2.0                                   # how much the level moves (Q), how noisy the sensor is (H)
GAP = range(60, 70)                                  # t = 60..69: no reading arrives / 읽기가 없는 구간
t_axis = np.arange(1, len(READING) + 1)
y_obs = np.array([np.nan if v is None else v for v in READING], dtype=float)
a, P = 0.0, 10.0                                     # start: a rough guess with a large variance
est, var, gain = [], [], []
for y_t in y_obs:
    a_pred, P_pred = a, P + Qv                       # predict: the level may have moved
    if np.isnan(y_t):
        a, P, K = a_pred, P_pred, np.nan             # no reading: skip the update
    else:
        K = P_pred / (P_pred + Hv)                   # gain
        a = a_pred + K * (y_t - a_pred)              # move K of the way toward the reading
        P = (1 - K) * P_pred
    est.append(a); var.append(P); gain.append(K)
est, var, gain = np.array(est), np.array(var), np.array(gain)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(t_axis, y_obs, '.', color='0.6', label='reading')
ax[0].plot(t_axis, LEVEL, color='black', lw=1, label='hidden level')
ax[0].plot(t_axis, est, color='#285943', label='filter estimate')
ax[0].fill_between(t_axis, est - 2 * np.sqrt(var), est + 2 * np.sqrt(var), color='#285943', alpha=.15, label='estimate ± 2√P')
ax[1].plot(t_axis, var, color='#b38b2d', label='variance P')
ax[1].plot(t_axis, gain, 'o', ms=3, color='#285943', label='gain K')
for a_ in ax:
    a_.axvspan(59.5, 69.5, color='0.88'); a_.set_xlabel('t'); a_.legend(fontsize=8)
ax[0].set_title('Readings, hidden level and filter estimate'); ax[1].set_title('Variance P and gain K (gray: no reading)')
plt.tight_layout(); plt.show()
pick = lambda ts: pd.DataFrame({'t': ts, 'gain_K': [gain[t - 1] for t in ts], 'variance_P': [var[t - 1] for t in ts]})
table('Q5_READ gain', pick([1, 2, 3, 5, 10, 59]))
table('Q5_READ variance', pick([59, 60, 65, 69, 70, 71, 75]))
print('NaN gain = no reading at that t, so no update. P is the variance of the error of the estimate about the hidden level.')

## Analysis Receipt
AI 도구·목적·범위를 적으세요. prior_holdout_access에는 미리 보지 않았다면 none, 봤다면 경위를 적으세요. 영수증 블록(===== 줄 사이)만 PDF에 복사 가능한 글자로 넣습니다. 그 아래 기계 기록은 IPYNB에 남으므로 PDF에 넣지 않습니다. / Copy only the receipt block into your PDF; the full machine record stays in your IPYNB. 자동 점검은 서술 답안의 정확성이나 사전 열람 여부를 인증하지 않습니다.

In [ ]:
#@title Analysis Receipt / 이 블록을 PDF에 복사 / Copy this block into your PDF
ai_use = "" #@param {type:"string"}
prior_holdout_access = "" #@param {type:"string"}
# Enter "none" if there was no earlier viewing; otherwise explain it honestly.
FUNCTIONS=[standardize,next_variance,forecast_variance,var_one_step,impulse_path,error_metrics]
fingerprints={str(i):hashlib.sha256(f.__code__.co_code+repr(f.__code__.co_consts).encode()).hexdigest()[:16] for i,f in enumerate(FUNCTIONS,1)}
consistent=commit_matches and all(a>b for a,b in AUDIT) and hashlib.sha256(DATA_TEXT.encode()).hexdigest()==EXPECTED_HASH and all(fingerprints[str(i)]==CHECK_SIGNATURES.get(i) for i in range(1,7))
ready=consistent and OPENED and all(CHECKS.get(i,False) for i in range(1,7)) and bool(ai_use.strip()) and bool(prior_holdout_access.strip()) and ('Q5_READ gain' in RESULTS)
receipt={'notebook':'A3 v8','Consistency':'OK' if consistent else 'NOT OK','Submission':'READY' if ready else 'NOT READY','coding_credit':int(all(CHECKS.get(i,False) for i in range(1,7))), 'checks':CHECKS,'commitment':lock,'current_choice':commit,'data_sha256':EXPECTED_HASH,'code_fingerprints':fingerprints,'run_utc':RUN_ID,'environment':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'arch':arch.__version__,'statsmodels':statsmodels.__version__,'runtime':'CPU'},'AI_use':ai_use,'prior_holdout_access':prior_holdout_access,'scope':'Weeks 9-11: GARCH, VAR, Kalman','results':RESULTS}
receipt['receipt_sha256']=digest(receipt)
from io import StringIO
def _t(name):
    return pd.read_json(StringIO(RESULTS[name]),orient='split') if name in RESULTS else None
def _p(v):
    return '< 0.001' if float(v) < 0.001 else f'{float(v):.3f}'
todo=[]
if not commit_matches: todo.append('restore your original choice, reason and ID / 원래 선택으로 되돌리기')
elif not OPENED: todo.append('choose, reason, ID, then reveal / 선택·이유·학번 뒤 공개')
miss=[str(i) for i in range(1,7) if not CHECKS.get(i,False)]
if miss: todo.append('Code '+','.join(miss)+' not passed: coding point only / 코드 점수만 영향')
if 'Q5_READ gain' not in RESULTS: todo.append('run Q5 / Q5 실행')
if not ai_use.strip(): todo.append('AI use / AI 사용란')
if not prior_holdout_access.strip(): todo.append('prior access / 사전 열람란')
L=[]
L.append(f"notebook         A3 v8    student ID {commit['student_id'] or '-'}")
L.append('consistency      '+('OK' if consistent else 'NOT OK: Runtime > Run all and keep your original choice'))
L.append('submission       '+('READY' if ready else 'NOT READY'))
if not ready:
    for t in todo: L.append('  to do          '+t)
L.append('code checks      '+' '.join(f"{i}:{'PASS' if CHECKS.get(i,False) else '-'}" for i in range(1,7))+f"    coding point {receipt['coding_credit']}")
d=_t('Q1_READ diagnostics')
if d is not None:
    g=d.set_index('series')
    L.append(f"Q1 lag-1 ACF     r {g.loc['r','acf1']:.3f} | r2 {g.loc['r2','acf1']:.3f}    residual p: z {_p(g.loc['z','p'])} | z2 {_p(g.loc['z2','p'])}")
v=_t('Q1_READ variance')
if v is not None:
    w=v.set_index('item')['variance_pct_squared']
    L.append(f"Q1 variance      next day {w['next day']:.3f} | day 10 {w['day 10']:.3f} | long-run {w['long-run']:.3f}  (squared % points)")
q=_t('Q2_READ')
if q is not None:
    r0=q.iloc[0]
    L.append(f"Q2 VAR1          stable {bool(r0['stable'])} | residual p {_p(r0['residual_p'])} | Granger x to y p {_p(r0['x_to_y_Granger_p'])}")
o=_t('Q3_READ same-time ordering')
if o is not None:
    oo=o.set_index('order')['x_response']
    L.append(f"Q3 same time     x response to the y shock: x first {oo['x first']:.3f} | y first {oo['y first']:.3f}")
sel=_t('Q4_SELECT y only')
if sel is not None:
    L.append('Q4 validation    y RMSE '+' | '.join(f"{m} {e:.3f}" for m,e in zip(sel['model'],sel['RMSE'])))
L.append(f"Q4 choice        {commit['preferred']}    reason: {(commit['reason'][:55]+'...') if len(commit['reason'])>55 else (commit['reason'] or '-')}")
fin=_t('Q4_FINAL y only')
if OPENED and fin is not None:
    L.append('Q4 final         y RMSE '+' | '.join(f"{m} {e:.3f}" for m,e in zip(fin['model'],fin['RMSE']))+f"    opened {lock['opened_utc'][:16]}")
    gc=_t('GARCH future comparison')
    if gc is not None:
        L.append(f"Q4 GARCH 10 days forecast {gc['forecast_variance'].min():.2f} to {gc['forecast_variance'].max():.2f} | squared shocks {gc['observed_squared_shock'].min():.2f} to {gc['observed_squared_shock'].max():.2f}")
else:
    L.append('Q4 final         not opened yet / 아직 열지 않음')
q5g=_t('Q5_READ gain'); q5v=_t('Q5_READ variance')
if q5g is not None:
    g5=q5g.set_index('t')['gain_K']
    L.append(f"Q5 gain K        t=1 {g5[1]:.3f} | t=10 {g5[10]:.3f} | t=59 {g5[59]:.3f}")
if q5v is not None:
    pv=q5v.set_index('t')['variance_P']
    L.append(f"Q5 variance P    t=59 {pv[59]:.3f} | t=69 {pv[69]:.3f} (no reading) | t=70 {pv[70]:.3f} | t=75 {pv[75]:.3f}")
L.append(f"AI use           {ai_use.strip() or '(empty)'}")
L.append(f"prior access     {prior_holdout_access.strip() or '(empty)'}")
L.append(f"environment      Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | arch {arch.__version__} | statsmodels {statsmodels.__version__}")
L.append(f"record hash      {receipt['receipt_sha256'][:16]}    full record: next cell and your IPYNB")
print('===== ANALYSIS RECEIPT: copy from here into your PDF / 여기부터 PDF에 복사 =====')
for line in L: print(line)
print('===== END OF RECEIPT / 여기까지 =====')
print('READY checks notebook completion, not the written answers or honesty. Submit PDF + IPYNB even if NOT READY. A new run is needed after any edit.')

In [ ]:
#@title 기계 기록 / Machine record. PDF에 복사하지 않습니다 / Do not copy into your PDF
# 채점자가 IPYNB 에서 확인하는 전체 기록입니다. 위 영수증의 record hash 와 같은 기록입니다.
# The grader checks this full record in your IPYNB. It is the record behind the record hash above.
print(json.dumps(receipt,ensure_ascii=False,indent=2))